# K2 — Partially labelled four-chamber federation over cached frozen features

Implementation of experiment **K2** from *Kaggle Experiment Designs* (Sep 29, 2026). One notebook holds the stage chain
`10_fed_harmonise_cache → 11_fed_baselines → 12_fed_method → 13_fed_privacy → 14_fed_student_stats`; each Kaggle session
runs one stage, reads the previous stage's output dataset and writes its own to `/kaggle/working`.

**Research question.** When three real echo datasets with different chamber label sets federate, does treating missing
structures as unlabelled (marginal loss plus conditional distillation) and aggregating only light adapters and decoders on a
frozen, licence-clean encoder produce a four-chamber model that beats every client's local model, at a fraction of the
communication of full-model federation, with privacy accounted for at several ε?

| Client | Labels held | Treated as unlabelled | Test |
| --- | --- | --- | --- |
| C1 CAMUS (GE Vivid E95) | LV cavity, myocardium, LA | RV, RA | CAMUS test split (LV, LA) |
| C2 HMC-QU | LV wall (myocardium) | LV cavity, RV, LA, RA | client test split |
| C3 CardiacNet-PAH | LV, RV, LA, RA | myocardium | CardiacNet held-out fold (RV, RA are the new numbers) |

EchoNet-Dynamic is excluded (research-use agreement forbids redistribution), so the client set differs from FedCVD's and every
baseline is re-run under this protocol rather than copied.

| Stage | Session | Work | Output dataset |
| --- | --- | --- | --- |
| 10 | S10 | harmonised label space, annotated-frame index, split manifests (+ hash), frozen-encoder feature caches at 1/8 and 1/16 (fp16), 112 px raw frames for the U-Net reference and the student | `fed-features-v1` |
| 11 | S11 | local-only per client; centralised pooled with background labelling and with marginal loss; FedAvg, FedProx, SCAFFOLD, FedInit, Fed-Consist, FedPSL-style head aggregation (all with background labelling) × seeds × rounds | `fed-baselines-v1` |
| 12 | S12 | FedAvg + marginal loss; + conditional distillation (ConDistFL); + FedBN; frozen-encoder swap (DINOv3 vs PanEcho); full-model 2-D U-Net FedAvg on raw frames as the communication reference | `fed-method-v1` |
| 13 | S13 | DP-SGD (Opacus, RDP accountant, δ = 1/N_client, stated clipping norm) at ε ∈ {1, 4, 8} for the two best configurations; simulated secure aggregation; membership-inference audit (loss-threshold and shadow); per-client morphological conformal margins | `fed-privacy-v1` |
| 14 | S14 | ≤10 M student distilled from the federated model; RV/LV area ratio from federated masks vs the RHC label; paired cluster bootstrap statistics; withdrawal conditions; figures | `k2-final-v1` |

**Running without the data (validation mode).** When `/kaggle/input` is absent or `SMOKE_TEST=1`, three synthetic clients
with exactly the label sets above are generated, a frozen random two-scale encoder stands in for DINOv3 / PanEcho, and every
stage runs end to end on CPU. This validates the code paths, not the science.

**To confirm on Kaggle before the first real session:** dataset layouts and label ids in `iter_camus`, `iter_hmcqu`,
`iter_cardiacnet`; the DINOv3 / PanEcho weight datasets expected by `DINOv3Encoder` / `PanEchoEncoder`.


In [1]:
# ---------------------------------------------------------------- Cell 1: configuration and environment
import os, sys, re, json, time, math, glob, copy, pickle, hashlib, random, shutil, platform, warnings, itertools
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F

warnings.filterwarnings("ignore", category=UserWarning); warnings.filterwarnings("ignore", category=FutureWarning)
torch.backends.cudnn.benchmark = True

ON_KAGGLE = Path("/kaggle/input").exists()
SMOKE_TEST = os.environ.get("SMOKE_TEST", "0" if ON_KAGGLE else "1") == "1"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPUS = torch.cuda.device_count()


def _env_list(name, default, cast=int):
    v = os.environ.get(name)
    return list(default) if v is None else [cast(x) for x in v.split(",") if x]


# unified label space: every client's ground truth is remapped into it; structures a client does not label become 0 ("other")
CLASSES = ("background", "LV", "MYO", "RV", "LA", "RA")
CID = {n: i for i, n in enumerate(CLASSES)}
CLIENT_LABELS = {"CAMUS": ["LV", "MYO", "LA"], "HMCQU": ["MYO"], "CARDIACNET": ["LV", "RV", "LA", "RA"]}
CLIENT_VENDOR = {"CAMUS": "GE Vivid E95", "HMCQU": "Philips/GE (mixed)", "CARDIACNET": "Philips/Hitachi"}
HELDOUT_STRUCTURES = {"CARDIACNET": ["RV", "RA"], "CAMUS": ["LV", "LA"]}


@dataclass
class K2Config:
    stages: List[int] = field(default_factory=lambda: _env_list("K2_STAGES", [10, 11, 12, 13, 14]))
    seeds: List[int] = field(default_factory=lambda: _env_list("K2_SEEDS", [0, 1, 2]))
    encoders: List[str] = field(default_factory=lambda: _env_list("K2_ENCODERS", ["dinov3", "panecho"], str))
    kaggle_input: Path = Path("/kaggle/input")
    work: Path = Path("/kaggle/working") if ON_KAGGLE else Path("./k2_work")
    camus_root: Path = Path(os.environ.get("CAMUS_ROOT", "/kaggle/input/camus"))
    hmcqu_root: Path = Path(os.environ.get("HMCQU_ROOT", "/kaggle/input/hmc-qu"))
    cardiacnet_root: Path = Path(os.environ.get("CARDIACNET_ROOT", "/kaggle/input/cardiacnet"))
    k1_cache: str = "cardiacnet-cache-v1"                       # K1's manifest (acquisition-grouped folds) is reused when attached
    dinov3_dir: Path = Path(os.environ.get("DINOV3_DIR", "/kaggle/input/dinov3-weights"))
    panecho_dir: Path = Path(os.environ.get("PANECHO_DIR", "/kaggle/input/panecho-weights"))
    cardiacnet_label_map: Dict[int, str] = field(default_factory=lambda: {1: "LV", 2: "RV", 3: "LA", 4: "RA"})   # VERIFY
    camus_label_map: Dict[int, str] = field(default_factory=lambda: {1: "LV", 2: "MYO", 3: "LA"})
    cardiacnet_time_axis: int = -1
    size: int = 224                                              # frame side for features and labels
    unet_size: int = 112                                         # FedCVD architecture reference resolution
    heldout_fold: int = 0
    cal_frac: float = 0.15
    test_frac: float = 0.15
    # federation protocol
    rounds: int = 100
    local_epochs: int = 2
    batch: int = 16
    lr: float = 1e-3
    eval_every: int = 5
    fedprox_mu: float = 0.01
    fedinit_beta: float = 0.1
    condist_weight: float = 1.0
    consist_weight: float = 0.5
    # privacy
    dp_eps: List[float] = field(default_factory=lambda: [1.0, 4.0, 8.0])
    dp_clip: float = 1.0
    # student and statistics
    student_epochs: int = 20
    n_boot: int = 2000
    coverage: float = 0.90
    conformal_max_margin: int = 12
    ds_feats_prefix: str = "fed-features"                       # one dataset version per encoder: fed-features-{encoder}-v1 (D2)
    ram_gb: float = 12.0
    camus_test_patients: Tuple[int, int] = (451, 500)            # official CAMUS 450/50 split (D1); a 'testing/' folder also marks test patients
    dp_unit: str = "video"                                       # DP adjacency unit (C3): one sample = one video with dp_frames_per_video frames
    dp_frames_per_video: Dict[str, int] = field(default_factory=lambda: {"CARDIACNET": 5, "CAMUS": 4, "HMCQU": 4})
    dp_videos_per_step: int = 8
    student_rounds: int = 30
    ds_base: str = "fed-baselines-v1"
    ds_method: str = "fed-method-v1"
    ds_priv: str = "fed-privacy-v1"
    ds_final: str = "k2-final-v1"


CFG = K2Config()
if SMOKE_TEST:
    CFG.size, CFG.unet_size, CFG.rounds, CFG.local_epochs, CFG.batch, CFG.eval_every = 64, 64, 12, 2, 8, 3
    CFG.seeds = _env_list("K2_SEEDS", [0, 1]); CFG.student_epochs, CFG.n_boot, CFG.conformal_max_margin, CFG.student_rounds, CFG.dp_videos_per_step = 2, 200, 6, 4, 4
CFG.work.mkdir(parents=True, exist_ok=True)
N_CLS = len(CLASSES)


def code_hash() -> str:
    for p in ["__notebook_source__.ipynb"] + sorted(glob.glob("*.ipynb")):
        if Path(p).exists(): return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:12]
    return hashlib.sha256(json.dumps(asdict(CFG), default=str, sort_keys=True).encode()).hexdigest()[:12]


ENV_INFO = dict(kaggle_image=os.environ.get("KAGGLE_DOCKER_IMAGE", os.environ.get("KAGGLE_CONTAINER_NAME", "local")), python=platform.python_version(),
                torch=torch.__version__, device=str(DEVICE), gpu=(torch.cuda.get_device_name(0) if N_GPUS else "none"), n_gpus=N_GPUS, code_hash=code_hash())


def stage_dir(name: str, create: bool = False) -> Path:
    local = CFG.work / name
    if local.exists() and any(local.iterdir()): return local
    attached = CFG.kaggle_input / name
    if attached.exists(): return attached
    if create:
        local.mkdir(parents=True, exist_ok=True); return local
    raise FileNotFoundError(f"artefact '{name}' not found in {CFG.work} or {CFG.kaggle_input}; run the producing stage or attach it")


def ds_feats(encoder: str) -> str:
    return f"{CFG.ds_feats_prefix}-{encoder}-v1"


FEATS_ALL = [ds_feats(e) for e in CFG.encoders]
STAGE_DEPS = {10: [], 11: [ds_feats(CFG.encoders[0])], 12: FEATS_ALL + [CFG.ds_base], 13: [ds_feats(CFG.encoders[0]), CFG.ds_base, CFG.ds_method], 14: [ds_feats(CFG.encoders[0]), CFG.ds_base, CFG.ds_method, CFG.ds_priv]}


def require_inputs(stage: int) -> None:
    deps = STAGE_DEPS[stage]; print(f"[stage {stage}] depends on: {deps or 'nothing'}")
    for d in deps:
        p = stage_dir(d); v = p / "VERSION.json"
        if not v.exists(): raise RuntimeError(f"{d} at {p} has no VERSION.json — the producing stage did not finish")
        print(f"   {d:20s} -> {p}  ({json.loads(v.read_text()).get('created')})")


def write_version(name: str, extra: Optional[dict] = None) -> Path:
    p = stage_dir(name, create=True)
    (p / "VERSION.json").write_text(json.dumps({"name": name, "created": time.strftime("%Y-%m-%dT%H:%M:%S"), "smoke": SMOKE_TEST, **ENV_INFO, **(extra or {})}, indent=1, default=str))
    return p


print(f"K2 | smoke={SMOKE_TEST} | device={DEVICE} ({ENV_INFO['gpu']}) | stages={CFG.stages} seeds={CFG.seeds} encoders={CFG.encoders} rounds={CFG.rounds} | work={CFG.work.resolve()}")

K2 | smoke=True | device=cpu (none) | stages=[10, 11, 12, 13, 14] seeds=[0, 1] encoders=['dinov3', 'panecho'] rounds=12 | work=/tmp/claude-0/-home-user-ph/d8985a5b-cc87-5db9-b4e3-f560f4f33009/scratchpad/run_k2/k2_work


In [2]:
# ---------------------------------------------------------------- Cell 2: utilities
from scipy import ndimage, stats
from scipy.special import expit
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold


def seed_everything(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def peak_mem_mb() -> float:
    return torch.cuda.max_memory_allocated() / 2 ** 20 if torch.cuda.is_available() else float("nan")


def count_params(m: nn.Module) -> int:
    return sum(p.numel() for p in m.parameters())


class RunLog:
    """One CSV per run under <dir>/runs/ (metrics, timing, memory, code hash, image tag); read() concatenates them."""
    def __init__(self, root: Path):
        self.dir = Path(root) / "runs"; self.dir.mkdir(parents=True, exist_ok=True)
    def append(self, run_id: str, row: dict) -> None:
        row = {"run_id": run_id, **row, **{f"env_{k}": v for k, v in ENV_INFO.items()}, "smoke": SMOKE_TEST, "logged_at": time.strftime("%Y-%m-%dT%H:%M:%S")}
        pd.DataFrame([row]).to_csv(self.dir / f"{run_id}.csv", index=False)
    def read(self) -> pd.DataFrame:
        fs = sorted(self.dir.glob("*.csv"))
        return pd.concat([pd.read_csv(f) for f in fs], ignore_index=True) if fs else pd.DataFrame()


class RunManifest:
    def __init__(self, path: Path):
        self.path = Path(path); self.done_set = set(json.loads(self.path.read_text())) if self.path.exists() else set()
    def done(self, rid): return rid in self.done_set
    def mark(self, rid):
        self.done_set.add(rid); self.path.parent.mkdir(parents=True, exist_ok=True); self.path.write_text(json.dumps(sorted(self.done_set), indent=0))


def dice_binary(a, b) -> float:
    a = a.astype(bool); b = b.astype(bool); s = a.sum() + b.sum()
    return 1.0 if s == 0 else float(2.0 * np.logical_and(a, b).sum() / s)


def hausdorff(pred, gt) -> float:
    """HD95 (95th percentile of the symmetric surface distances) in PIXELS via distance transforms of the boundaries; nan if either mask is empty."""
    bp = pred & ~ndimage.binary_erosion(pred); bg = gt & ~ndimage.binary_erosion(gt)
    if bp.sum() == 0 or bg.sum() == 0: return float("nan")
    d = np.concatenate([ndimage.distance_transform_edt(~bg)[bp], ndimage.distance_transform_edt(~bp)[bg]])
    return float(np.percentile(d, 95))


def holm(pvals):
    p = np.asarray(pvals, float); m = len(p); order = np.argsort(p); adj = np.empty(m); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i]); adj[i] = min(1.0, run)
    return adj.tolist()


def autocast_ctx():
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE.type == "cuda"))


class Timer:
    def __enter__(self): self.t = time.time(); return self
    def __exit__(self, *a): self.dt = time.time() - self.t

In [3]:
# ---------------------------------------------------------------- Cell 3: data — harmonised clients (synthetic in validation mode; CAMUS / HMC-QU / CardiacNet loaders on Kaggle)
# A frame record: dict(client, id (frame id), video (patient/acquisition group for splitting), frame uint8 [H,W], label uint8 [H,W] in the unified
# space with the client's unlabelled structures set to 0, pah (0/1 or nan; CardiacNet only), view, ed_es)

def _ellipse(H, W, cy, cx, ry, rx):
    yy, xx = np.mgrid[0:H, 0:W]
    return (((yy - cy) / max(ry, 1e-3)) ** 2 + ((xx - cx) / max(rx, 1e-3)) ** 2) <= 1.0


def synth_frame(rng, H, W, phase_con, pah=0, style=None, two_chamber=False):
    """Cartoon apical view with full six-class truth (bg, LV, MYO, RV, LA, RA); clients then drop the structures they do not label."""
    style = style or dict(gain=rng.uniform(0.7, 1.3), offset=rng.uniform(-0.1, 0.1), speckle=rng.uniform(0.05, 0.2), shift=rng.integers(-4, 5, 2))
    rv_scale = 0.75 + 0.45 * pah + rng.normal(0, 0.1); v = 0.82 + 0.18 * phase_con; a = 1.0 - 0.15 * phase_con
    cy0, cx0 = H * 0.5 + style["shift"][0], W * 0.5 + style["shift"][1]
    lab = np.zeros((H, W), np.uint8)
    lv = _ellipse(H, W, cy0 + 0.13 * H, cx0 + (0.18 if not two_chamber else 0.0) * W, 0.24 * H * v, 0.13 * W * v)
    la = _ellipse(H, W, cy0 - 0.20 * H, cx0 + (0.16 if not two_chamber else 0.0) * W, 0.11 * H * a, 0.11 * W * a)
    lab[lv] = CID["LV"]; lab[la] = CID["LA"]
    if not two_chamber:
        rv = _ellipse(H, W, cy0 + 0.11 * H, cx0 - 0.16 * W, 0.22 * H * v * rv_scale, 0.12 * W * v * rv_scale)
        ra = _ellipse(H, W, cy0 - 0.20 * H, cx0 - 0.15 * W, 0.10 * H * a, 0.10 * W * a)
        lab[rv] = CID["RV"]; lab[ra] = CID["RA"]
    myo = ndimage.binary_dilation(lv, iterations=max(2, H // 24)) & (lab == 0); lab[myo] = CID["MYO"]
    wall = ndimage.binary_dilation(lab > 0, iterations=3) & (lab == 0)
    yy = np.mgrid[0:H, 0:W][0]; sector = _ellipse(H, W, 0, cx0, H * 1.05, W * 0.62) & (yy > 2)
    img = 0.25 + 0.55 * (wall | myo) - 0.15 * ((lab > 0) & (lab != CID["MYO"])) + 0.05 * np.sin(yy / 6.0)
    img = img * style["gain"] + style["offset"] + rng.gamma(4.0, style["speckle"] / 4.0, (H, W)) - style["speckle"]
    img = ndimage.gaussian_filter(img, 0.8) * sector
    return (np.clip(img, 0, 1) * 255).astype(np.uint8), lab


def restrict_labels(lab: np.ndarray, client: str) -> np.ndarray:
    keep = [CID[n] for n in CLIENT_LABELS[client]]; out = np.zeros_like(lab)
    for c in keep: out[lab == c] = c
    return out


def synth_client(client: str, n_videos: int, seed: int, H: int, W: int) -> List[dict]:
    rng = np.random.default_rng(seed); recs = []
    for v in range(n_videos):
        style = dict(gain=rng.uniform(0.7, 1.3) * (1.15 if client == "CAMUS" else 1.0), offset=rng.uniform(-0.1, 0.1), speckle=rng.uniform(0.05, 0.2), shift=rng.integers(-4, 5, 2))
        pah = int(rng.random() < 0.65) if client == "CARDIACNET" else 0
        if client == "CAMUS":      # ED/ES × A4C/A2C
            frames = [(f"{view}_{ph}", con, view == "2CH") for view in ("4CH", "2CH") for ph, con in (("ED", 1.0), ("ES", 0.0))]
        elif client == "HMCQU":    # a few annotated A4C frames over one cycle
            frames = [(f"f{t}", 0.5 + 0.5 * np.cos(2 * np.pi * t / 6), False) for t in range(4)]
        else:                      # CardiacNet: five annotated frames per video
            frames = [(f"f{t}", 0.5 + 0.5 * np.cos(2 * np.pi * t / 5 + 0.3), False) for t in range(5)]
        for name, con, two in frames:
            img, lab = synth_frame(rng, H, W, con, pah, style, two)
            recs.append(dict(client=client, id=f"{client}_v{v:03d}_{name}", video=f"{client}_v{v:03d}", frame=img, label=restrict_labels(lab, client), pah=(pah if client == "CARDIACNET" else np.nan), view=name.split("_")[0], full_label=lab,
                             is_test=(client == "CAMUS" and v >= n_videos - 2)))     # synthetic stand-in for the official CAMUS test patients
    return recs


# ---------------- real loaders --------------------------------------------------------------------------------------------
def _to_uint8(a):
    a = a.astype(np.float32); lo, hi = np.percentile(a, [0.5, 99.5]); return (np.clip((a - lo) / max(hi - lo, 1e-6), 0, 1) * 255).astype(np.uint8)


def _read_volume(path: Path) -> np.ndarray:
    s = str(path)
    if s.endswith((".nii", ".nii.gz")):
        import nibabel as nib
        a = np.asanyarray(nib.load(s).dataobj).squeeze()
        return a if a.ndim == 2 else np.moveaxis(a, CFG.cardiacnet_time_axis, 0)
    if s.endswith(".mhd"):
        import SimpleITK as sitk
        return sitk.GetArrayFromImage(sitk.ReadImage(s)).squeeze()
    raise ValueError(s)


def _resize(img, size, nearest=False):
    t = torch.from_numpy(np.ascontiguousarray(img)).float()[None, None]
    return F.interpolate(t, size=(size, size), mode="nearest" if nearest else "bilinear", **({} if nearest else {"align_corners": False}))[0, 0].numpy()


def pad_square(img: np.ndarray, label: Optional[np.ndarray] = None):
    """Pad to a square with the sector background value (median of the border pixels) before resizing, so frames are never squashed (D3);
    labels are padded with 0 and resized with nearest neighbour by the caller."""
    H, W = img.shape; S = max(H, W); bg = float(np.median(np.concatenate([img[0], img[-1], img[:, 0], img[:, -1]])))
    top, left = (S - H) // 2, (S - W) // 2
    out = np.full((S, S), bg, img.dtype); out[top:top + H, left:left + W] = img
    if label is None: return out, None
    lo = np.zeros((S, S), label.dtype); lo[top:top + H, left:left + W] = label
    return out, lo


def iter_camus(root: Path) -> List[dict]:
    """CAMUS: <root>/**/patientXXXX/patientXXXX_{2CH,4CH}_{ED,ES}(.nii.gz|.mhd) with *_gt files; labels 1 LV endo, 2 myocardium, 3 LA."""
    recs = []
    for gt in sorted(list(Path(root).rglob("patient*_gt.nii.gz")) + list(Path(root).rglob("patient*_gt.mhd"))):
        m = re.match(r"(patient\d+)_(2CH|4CH)_(ED|ES)_gt", gt.name)
        if not m: continue
        img = gt.with_name(gt.name.replace("_gt", ""))
        if not img.exists(): continue
        lab_raw = _read_volume(gt).astype(np.int64); lab = np.zeros(lab_raw.shape, np.uint8)
        for raw, name in CFG.camus_label_map.items(): lab[lab_raw == raw] = CID[name]
        pnum = int(m[1][7:]); is_test = ("testing" in [q.lower() for q in gt.parts]) or (CFG.camus_test_patients[0] <= pnum <= CFG.camus_test_patients[1])
        recs.append(dict(client="CAMUS", id=f"CAMUS_{m[1]}_{m[2]}_{m[3]}", video=f"CAMUS_{m[1]}", frame=_to_uint8(_read_volume(img)), label=restrict_labels(lab, "CAMUS"), pah=np.nan, view=m[2], is_test=is_test))
    return recs


def iter_hmcqu(root: Path) -> List[dict]:
    """HMC-QU: A4C videos (*.avi) and LV-wall masks in 'LV Ground-truth Segmentation Masks/Mask_<name>.mat' (variable 'predicted',
    one binary mask per frame from the ED frame listed in A4C.xlsx). Masks are 224×224; frames are resized to match."""
    import cv2
    from scipy.io import loadmat
    recs = []; xl = list(Path(root).rglob("A4C*.xlsx")); starts = {}
    if xl:
        df = pd.read_excel(xl[0]); name_col = [c for c in df.columns if "echo" in str(c).lower() or "name" in str(c).lower()]
        st_col = [c for c in df.columns if "start" in str(c).lower()]
        if name_col and st_col: starts = {str(r[name_col[0]]).replace(".avi", ""): int(r[st_col[0]]) for _, r in df.iterrows()}
    for mat in sorted(Path(root).rglob("Mask_*.mat")):
        name = mat.stem[5:]; vids = list(Path(root).rglob(f"{name}.avi"))
        if not vids: continue
        masks = loadmat(mat).get("predicted"); cap = cv2.VideoCapture(str(vids[0])); frames = []
        while True:
            ok, f = cap.read()
            if not ok: break
            frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2GRAY))
        cap.release(); s0 = starts.get(name, 1) - 1
        for k in range(masks.shape[0]):
            t = s0 + k
            if t >= len(frames): break
            lab = np.zeros(masks[k].shape, np.uint8); lab[masks[k] > 0] = CID["MYO"]
            recs.append(dict(client="HMCQU", id=f"HMCQU_{name}_f{t}", video=f"HMCQU_{name}", frame=_to_uint8(_resize(frames[t], masks.shape[1])), label=lab, pah=np.nan, view="4CH"))
    return recs


def iter_cardiacnet(root: Path) -> List[dict]:
    """CardiacNet-PAH: annotated frames (label volume non-zero) of every video; label ids per CFG.cardiacnet_label_map (VERIFY)."""
    recs = []
    for img in sorted(p for p in Path(root).rglob("*.nii*") if "label" not in p.name.lower() and "asd" not in str(p).lower()):
        labs = [q for q in img.parent.glob("*.nii*") if "label" in q.name.lower()]
        if not labs: continue
        parts = [p.lower() for p in img.parts]; y = 0 if any(re.fullmatch(r"non[-_ ]?pah", p) for p in parts) else (1 if "pah" in parts else np.nan)
        stem = img.parent.name if img.stem.startswith("image") else img.stem.split(".")[0]
        vol = _to_uint8(_read_volume(img)); lv = _read_volume(labs[0]).astype(np.int64)
        for t in range(vol.shape[0]):
            if not lv[t].any(): continue
            lab = np.zeros(lv[t].shape, np.uint8)
            for raw, name in CFG.cardiacnet_label_map.items(): lab[lv[t] == raw] = CID[name]
            recs.append(dict(client="CARDIACNET", id=f"CARDIACNET_{stem}_f{t}", video=f"CARDIACNET_{stem}", frame=vol[t], label=lab, pah=y, view="4CH"))
    return recs


def discover_clients() -> Dict[str, List[dict]]:
    if SMOKE_TEST:
        return {"CAMUS": synth_client("CAMUS", 14, 1, 96, 96), "HMCQU": synth_client("HMCQU", 12, 2, 96, 96), "CARDIACNET": synth_client("CARDIACNET", 16, 3, 96, 96)}
    out = {"CAMUS": iter_camus(CFG.camus_root), "HMCQU": iter_hmcqu(CFG.hmcqu_root), "CARDIACNET": iter_cardiacnet(CFG.cardiacnet_root)}
    for k, v in out.items():
        if not v: raise FileNotFoundError(f"client {k}: no annotated frames found — check the dataset attachment and loader layout")
    return out


def assign_splits(client: str, recs: List[dict], seed=0) -> pd.DataFrame:
    """Patient/acquisition-level train / cal / test split per client. CAMUS: the official 50 test patients (451–500 or the testing/ folder).
    CardiacNet: K1's held-out fold from cardiacnet-cache-v1 when attached, otherwise a stratified group fold. HMC-QU: grouped 15 % split.
    The calibration set (for conformal margins) is carved from the training portion."""
    df = pd.DataFrame([{k: r[k] for k in ("client", "id", "video", "pah", "view")} for r in recs]); df["split"] = "train"
    if client == "CAMUS":                                        # D1: the official 450/50 patient split; calibration inside the 450
        df.loc[[bool(r.get("is_test", False)) for r in recs], "split"] = "test"
    elif client == "CARDIACNET":
        k1 = CFG.kaggle_input / CFG.k1_cache / "manifest.csv"
        if k1.exists():
            man = pd.read_csv(k1); fold = {f"CARDIACNET_{r.id.split('_', 1)[1]}": r.fold for _, r in man.iterrows()}
            df.loc[df.video.map(fold).eq(CFG.heldout_fold), "split"] = "test"
        else:
            vids = df.drop_duplicates("video"); skf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
            te = list(skf.split(vids, vids.pah.fillna(0).astype(int), vids.video))[CFG.heldout_fold][1]
            df.loc[df.video.isin(vids.video.iloc[te]), "split"] = "test"
    else:
        tr, te = next(GroupShuffleSplit(n_splits=1, test_size=CFG.test_frac, random_state=seed).split(df, groups=df.video)); df.loc[df.index[te], "split"] = "test"
    trn = df[df.split == "train"]; _, ca = next(GroupShuffleSplit(n_splits=1, test_size=CFG.cal_frac, random_state=seed).split(trn, groups=trn.video))
    df.loc[trn.index[ca], "split"] = "cal"
    return df

In [4]:
# ---------------------------------------------------------------- Cell 4: frozen encoders and Stage 10 — harmonise, index, split manifests, feature caches (S10)
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
class SyntheticFrozenEncoder(nn.Module):
    """Frozen random two-scale CNN used in validation mode (stands in for DINOv3 ViT-S/16 and PanEcho ConvNeXt-T)."""
    name, c8, c16 = "synthetic", 96, 192
    def __init__(self):
        super().__init__(); torch.manual_seed(7)
        self.s2 = nn.Sequential(nn.Conv2d(1, 32, 5, 2, 2), nn.GELU()); self.s4 = nn.Sequential(nn.Conv2d(32, 64, 3, 2, 1), nn.GELU())
        self.s8 = nn.Sequential(nn.Conv2d(64, 96, 3, 2, 1), nn.GELU()); self.s16 = nn.Sequential(nn.Conv2d(96, 192, 3, 2, 1), nn.GELU())
        for p in self.parameters(): p.requires_grad_(False)
    def forward(self, x):
        a = self.s8(self.s4(self.s2(x - 0.5))); return a, self.s16(a)


class DINOv3Encoder(nn.Module):
    """DINOv3 ViT-S/16 (licence permits research). Expects a Kaggle dataset with the dinov3 repo (torch.hub, source='local') and the
    ViT-S/16 checkpoint. 1/16 features = patch tokens at the native resolution; 1/8 features = patch tokens of the 2× upsampled frame."""
    name, c8, c16 = "dinov3", 384, 384
    def __init__(self, d: Path):
        super().__init__()
        repo = next((p for p in [d / "dinov3", d] if (p / "hubconf.py").exists()), None); ck = next(iter(d.rglob("dinov3_vits16*.pth")), None)
        if repo is None or ck is None: raise FileNotFoundError(f"DINOv3 repo/hubconf or dinov3_vits16*.pth not found under {d}")
        self.m = torch.hub.load(str(repo), "dinov3_vits16", source="local", weights=str(ck)).eval()
        for p in self.m.parameters(): p.requires_grad_(False)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)); self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
    def _tok(self, x):
        return self.m.get_intermediate_layers((x.repeat(1, 3, 1, 1) - self.mean) / self.std, n=1, reshape=True)[0]
    def forward(self, x):
        return self._tok(F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)), self._tok(x)


class PanEchoEncoder(nn.Module):
    """PanEcho ConvNeXt-T image encoder (CC BY-NC-SA): timm features_only at strides 8 and 16; weights from the PanEcho release
    (state-dict keys are matched by suffix, so a 'backbone.'/'encoder.' prefix is tolerated)."""
    name, c8, c16 = "panecho", 192, 384
    def __init__(self, d: Path):
        super().__init__()
        import timm
        self.m = timm.create_model("convnext_tiny", pretrained=False, features_only=True, out_indices=(1, 2)); ck = next(iter(Path(d).rglob("*.pt*")), None)
        if ck is None: raise FileNotFoundError(f"no PanEcho checkpoint under {d}")
        sd = torch.load(ck, map_location="cpu", weights_only=False); sd = sd.get("state_dict", sd.get("model", sd)); own = self.m.state_dict()
        matched = {k: next((v for kk, v in sd.items() if kk.endswith(k) and v.shape == own[k].shape), None) for k in own}
        self.n_matched, self.n_total = sum(v is not None for v in matched.values()), len(own)
        print(f"[PanEcho] matched {self.n_matched}/{self.n_total} ConvNeXt-T tensors")
        assert self.n_matched == self.n_total, "PanEcho checkpoint does not cover every ConvNeXt-T tensor — aborting (D7)"
        self.m.load_state_dict(matched); self.m.eval()
        for p in self.m.parameters(): p.requires_grad_(False)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)); self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
    def forward(self, x):
        f8, f16 = self.m((x.repeat(1, 3, 1, 1) - self.mean) / self.std); return f8, f16


def build_encoder(name: str) -> nn.Module:
    if SMOKE_TEST: e = SyntheticFrozenEncoder(); e.name = name           # the same stand-in serves both names in validation mode
    elif name == "dinov3": e = DINOv3Encoder(CFG.dinov3_dir)
    elif name == "panecho": e = PanEchoEncoder(CFG.panecho_dir)
    else: raise ValueError(name)
    return e.to(DEVICE).eval()


def load_manifest(encoder: Optional[str] = None):
    d = stage_dir(ds_feats(encoder or CFG.encoders[0])); man = pd.read_csv(d / "manifest.csv")
    assert (d / "manifest.sha256").read_text() == hashlib.sha256((d / "manifest.csv").read_bytes()).hexdigest(), "manifest hash mismatch"
    return d, man


@torch.no_grad()
def encoder_selftest(enc: nn.Module, frame: np.ndarray, out_png: Path) -> dict:
    """D7: forward one real frame, print feature shapes, save a PCA-3 false-colour image of the 1/8 features for a visual sanity check."""
    from sklearn.decomposition import PCA
    x = torch.from_numpy(frame).float()[None, None].to(DEVICE) / 255.0
    with autocast_ctx(): s8, s16 = enc(x)
    f = s8[0].float().cpu().numpy().reshape(s8.shape[1], -1).T; pcs = PCA(3).fit_transform(f); pcs = (pcs - pcs.min(0)) / (np.ptp(pcs, axis=0) + 1e-6)
    fig, ax = plt.subplots(1, 2, figsize=(7, 3.5)); ax[0].imshow(frame, cmap="gray"); ax[0].set_title("frame"); ax[1].imshow(pcs.reshape(s8.shape[-2], s8.shape[-1], 3)); ax[1].set_title(f"{enc.name} 1/8 features, PCA-3")
    for a in ax: a.axis("off")
    plt.tight_layout(); plt.savefig(out_png, dpi=90); plt.close()
    info = dict(s8=list(s8.shape[1:]), s16=list(s16.shape[1:]), matched=f"{getattr(enc, 'n_matched', 'n/a')}/{getattr(enc, 'n_total', 'n/a')}")
    print(f"[stage 10] self-test {enc.name}: s8 {tuple(s8.shape[1:])}, s16 {tuple(s16.shape[1:])}, matched tensors {info['matched']} → {out_png.name}")
    return info


@torch.no_grad()
def stage10_harmonise_and_cache():
    """Harmonised labels, official/grouped splits, square-padded frames and labels as ONE uint8 memmap each, and per-encoder fp16 memmaps
    (s8.npy, s16.npy) with a row index — one dataset version per encoder (D2, D3)."""
    require_inputs(10)
    clients = discover_clients(); mans = []
    for name, recs in clients.items():
        df = assign_splits(name, recs); df["labelled"] = json.dumps(CLIENT_LABELS[name]); df["orig_h"] = [r["frame"].shape[0] for r in recs]; df["orig_w"] = [r["frame"].shape[1] for r in recs]; mans.append(df)
        print(f"[stage 10] {name:11s}: {len(df):5d} frames / {df.video.nunique():4d} videos | labels {CLIENT_LABELS[name]} | splits {df.split.value_counts().to_dict()}")
    man = pd.concat(mans, ignore_index=True); man["row"] = np.arange(len(man)); all_recs = [r for recs in clients.values() for r in recs]
    frames = np.zeros((len(man), CFG.size, CFG.size), np.uint8); labels = np.zeros((len(man), CFG.size, CFG.size), np.uint8)
    for i, r in enumerate(all_recs):
        fr, lb = pad_square(r["frame"], r["label"])
        frames[i] = _resize(fr, CFG.size).round().clip(0, 255).astype(np.uint8); labels[i] = _resize(lb, CFG.size, nearest=True).astype(np.uint8)
    meta_all = {}
    for enc_name in CFG.encoders:
        out = stage_dir(ds_feats(enc_name), create=True); man.to_csv(out / "manifest.csv", index=False)
        (out / "manifest.sha256").write_text(hashlib.sha256((out / "manifest.csv").read_bytes()).hexdigest())
        np.save(out / "frames.npy", frames); np.save(out / "labels.npy", labels); man[["row", "id", "client", "video", "split"]].to_csv(out / "index.csv", index=False)
        enc = build_encoder(enc_name); t0 = time.time(); st = encoder_selftest(enc, frames[0], out / f"selftest_{enc_name}.png")
        with autocast_ctx(): s8, s16 = enc(torch.from_numpy(frames[:1]).float()[:, None].to(DEVICE) / 255.0)
        mm8 = np.lib.format.open_memmap(out / "s8.npy", mode="w+", dtype=np.float16, shape=(len(man), *s8.shape[1:]))
        mm16 = np.lib.format.open_memmap(out / "s16.npy", mode="w+", dtype=np.float16, shape=(len(man), *s16.shape[1:]))
        for b0 in range(0, len(man), 32):
            x = torch.from_numpy(frames[b0:b0 + 32]).float()[:, None].to(DEVICE) / 255.0
            with autocast_ctx(): a, b = enc(x)
            mm8[b0:b0 + len(x)] = a.half().cpu().numpy(); mm16[b0:b0 + len(x)] = b.half().cpu().numpy()
        mm8.flush(); mm16.flush(); del mm8, mm16
        meta = dict(c8=int(s8.shape[1]), c16=int(s16.shape[1]), h8=int(s8.shape[-1]), h16=int(s16.shape[-1]), encoder_class=type(enc).__name__, selftest=st, sec=time.time() - t0,
                    bytes_per_frame=int((np.prod(s8.shape[1:]) + np.prod(s16.shape[1:])) * 2), n_frames=int(len(man)))
        (out / "meta.json").write_text(json.dumps({enc_name: meta}, indent=1)); meta_all[enc_name] = meta
        write_version(ds_feats(enc_name), dict(encoder=enc_name, n_frames=int(len(man)), size=CFG.size, meta=meta, storage="fp16 memmap s8.npy/s16.npy + index.csv"))
        print(f"[stage 10] {enc_name}: s8 {tuple(s8.shape[1:])}, s16 {tuple(s16.shape[1:])} for {len(man)} frames → {out} ({(out/'s8.npy').stat().st_size/2**20 + (out/'s16.npy').stat().st_size/2**20:.0f} MB, {time.time()-t0:.0f}s)")
        del enc


if 10 in CFG.stages:
    stage10_harmonise_and_cache()

[stage 10] depends on: nothing


[stage 10] CAMUS      :    56 frames /   14 videos | labels ['LV', 'MYO', 'LA'] | splits {'train': 40, 'cal': 8, 'test': 8}
[stage 10] HMCQU      :    48 frames /   12 videos | labels ['MYO'] | splits {'train': 32, 'cal': 8, 'test': 8}
[stage 10] CARDIACNET :    80 frames /   16 videos | labels ['LV', 'RV', 'LA', 'RA'] | splits {'train': 55, 'test': 15, 'cal': 10}


[stage 10] self-test dinov3: s8 (96, 8, 8), s16 (192, 4, 4), matched tensors n/a/n/a → selftest_dinov3.png
[stage 10] dinov3: s8 (96, 8, 8), s16 (192, 4, 4) for 184 frames → k2_work/fed-features-dinov3-v1 (3 MB, 0s)
[stage 10] self-test panecho: s8 (96, 8, 8), s16 (192, 4, 4), matched tensors n/a/n/a → selftest_panecho.png


[stage 10] panecho: s8 (96, 8, 8), s16 (192, 4, 4) for 184 frames → k2_work/fed-features-panecho-v1 (3 MB, 0s)


In [5]:
# ---------------------------------------------------------------- Cell 5: models (feature-space decoder + adapters, U-Net reference, student) and label-mismatch losses
def make_norm(c, kind):
    return nn.BatchNorm2d(c) if kind == "bn" else nn.GroupNorm(8, c)


class Adapter(nn.Module):
    """Bottleneck residual adapter on a frozen feature map (zero-initialised up-projection)."""
    def __init__(self, c, r=4):
        super().__init__(); self.down = nn.Conv2d(c, c // r, 1); self.up = nn.Conv2d(c // r, c, 1); nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x): return x + self.up(F.gelu(self.down(x)))


class FeatDecoder(nn.Module):
    """Light FPN decoder (~1 M) with adapters (~0.2 M) over cached 1/8 and 1/16 features; six-class head (bg, LV, MYO, RV, LA, RA).
    Normalisation is BatchNorm (client-local under FedBN) or GroupNorm (DP-SGD arms)."""
    def __init__(self, c8, c16, mid=128, n_cls=N_CLS, norm="bn"):
        super().__init__()
        self.ad8, self.ad16 = Adapter(c8), Adapter(c16); self.lat8, self.lat16 = nn.Conv2d(c8, mid, 1), nn.Conv2d(c16, mid, 1)
        blk = lambda ci, co: nn.Sequential(nn.Conv2d(ci, co, 3, padding=1, bias=False), make_norm(co, norm), nn.ReLU())
        self.b16 = blk(mid, mid); self.b8 = nn.Sequential(blk(mid, mid), blk(mid, mid)); self.b4 = blk(mid, mid // 2); self.head = nn.Conv2d(mid // 2, n_cls, 1)
    def forward(self, s8, s16, out_hw):
        x16 = self.b16(self.lat16(self.ad16(s16)))
        x8 = self.b8(self.lat8(self.ad8(s8)) + F.interpolate(x16, size=s8.shape[-2:], mode="bilinear", align_corners=False))
        x4 = self.b4(F.interpolate(x8, scale_factor=2, mode="bilinear", align_corners=False))
        return F.interpolate(self.head(x4), size=out_hw, mode="bilinear", align_corners=False)
    def adapter_parameters(self): return list(self.ad8.parameters()) + list(self.ad16.parameters())


class UNet2D(nn.Module):
    """Plain 2-D U-Net (FedCVD's architecture family) on raw 112² frames: the full-model communication reference and, with base=16, the ≤10 M student."""
    def __init__(self, base=32, in_ch=1, n_cls=N_CLS):
        super().__init__()
        def blk(ci, co): return nn.Sequential(nn.Conv2d(ci, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(), nn.Conv2d(co, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU())
        ch = [base, base * 2, base * 4, base * 8, base * 16]
        self.enc = nn.ModuleList([blk(in_ch, ch[0])] + [blk(ch[i], ch[i + 1]) for i in range(4)])
        self.up = nn.ModuleList([nn.ConvTranspose2d(ch[i + 1], ch[i], 2, 2) for i in range(4)]); self.dec = nn.ModuleList([blk(ch[i] * 2, ch[i]) for i in range(4)])
        self.head = nn.Conv2d(ch[0], n_cls, 1)
    def forward(self, x):
        skips = []
        for i, e in enumerate(self.enc):
            x = e(x if i == 0 else F.max_pool2d(x, 2)); skips.append(x)
        x = skips[-1]
        for i in reversed(range(4)):
            x = self.dec[i](torch.cat([self.up[i](x), skips[i]], 1))
        return self.head(x)


# ---------------- losses under label mismatch ------------------------------------------------------------------------------
def soft_dice(logits, target, classes):
    p = logits.float().softmax(1); loss = 0.0
    for c in classes:
        pc, g = p[:, c], (target == c).float(); loss = loss + 1 - (2 * (pc * g).sum() + 1) / (pc.sum() + g.sum() + 1)
    return loss / max(1, len(classes))


def bg_ce(logits, target, labelled):
    """Background labelling: unlabelled structures are penalised as background (the FedCVD-style collapse case)."""
    return F.cross_entropy(logits.float(), target) + soft_dice(logits, target, labelled)


def marginal_ce(logits, target, labelled):
    """Marginal loss (ConDistFL): probabilities of every structure the client does not label are merged with background into one
    'other' class before the cross-entropy, so an unlabelled RV pixel in CAMUS is never penalised as background."""
    logp = logits.float().log_softmax(1); other = [c for c in range(N_CLS) if c not in labelled]
    collapsed = torch.cat([torch.logsumexp(logp[:, other], dim=1, keepdim=True), logp[:, labelled]], 1)
    remap = torch.zeros(N_CLS, dtype=torch.long, device=target.device)
    for i, c in enumerate(labelled): remap[c] = i + 1
    return F.nll_loss(collapsed, remap[target]) + soft_dice(logits, target, labelled)


def condist_loss(logits_local, logits_global, target, labelled):
    """Conditional distillation (ConDistFL): on pixels the client labels as 'other', the global model's conditional distribution over
    {background ∪ unlabelled structures} is the soft target for the local model (KL, temperature 1). Written as a masked mean so that it
    is also valid under torch.func.vmap (per-video DP gradients)."""
    other = [c for c in range(N_CLS) if c not in labelled]
    if len(other) < 2: return logits_local.sum() * 0.0
    region = (target == 0).float()
    q_g = logits_global.float()[:, other].softmax(1); logq_l = logits_local.float()[:, other].log_softmax(1)
    kl = (q_g * (q_g.clamp_min(1e-8).log() - logq_l)).sum(1)
    return (kl * region).sum() / (region.sum() + 1e-6)


def consistency_loss(logits_local_perturbed, logits_global):
    """Fed-Consist: prediction consistency between the local model on perturbed features and the global model (all pixels)."""
    return F.mse_loss(logits_local_perturbed.float().softmax(1), logits_global.float().softmax(1))


def transmitted_keys(model: nn.Module, fedbn: bool) -> List[str]:
    """State-dict keys that leave the client each round: everything except normalisation parameters/buffers under FedBN."""
    norm_prefixes = {n for n, m in model.named_modules() if isinstance(m, (nn.BatchNorm2d, nn.GroupNorm))}
    return [k for k in model.state_dict() if not (fedbn and any(k.startswith(p + ".") for p in norm_prefixes))]

In [6]:
# ---------------------------------------------------------------- Cell 6: federated engine — clients, local training (FedAvg/FedProx/SCAFFOLD/FedInit/Fed-Consist/FedPSL, marginal loss, ConDist, FedBN, video-level DP-SGD), aggregation, evaluation
class FrameStore:
    """Row-indexed access to the memmapped caches (frames.npy, labels.npy uint8; s8.npy, s16.npy fp16). Preloaded into RAM when the client's
    share is below CFG.ram_gb, read from the memmaps otherwise (D2)."""
    def __init__(self, rows: np.ndarray, root: Path, raw: bool):
        self.root, self.raw = Path(root), raw
        self.mm = {k: np.load(self.root / f"{k}.npy", mmap_mode="r") for k in ("frames", "labels", "s8", "s16")}
        per = sum(int(np.prod(self.mm[k].shape[1:])) * self.mm[k].dtype.itemsize for k in self.mm); self.rows = np.asarray(rows)
        self.ram = {k: np.ascontiguousarray(self.mm[k][self.rows]) for k in self.mm} if per * len(rows) < CFG.ram_gb * 2 ** 30 else None
        self.pos = {int(r): i for i, r in enumerate(self.rows)}
    def _take(self, k, rows):
        return self.ram[k][[self.pos[int(r)] for r in rows]] if self.ram is not None else np.stack([self.mm[k][int(r)] for r in rows])
    def batch(self, rows: List[int], device=DEVICE) -> dict:
        out = dict(s8=torch.from_numpy(self._take("s8", rows)).float().to(device), s16=torch.from_numpy(self._take("s16", rows)).float().to(device),
                   label=torch.from_numpy(self._take("labels", rows)).long().to(device))
        if self.raw:
            fr = torch.from_numpy(self._take("frames", rows)).float()[:, None] / 255.0
            out["frame"] = F.interpolate(fr, size=(CFG.unet_size, CFG.unet_size), mode="bilinear", align_corners=False).to(device)
            out["label_full"] = out["label"]; out["label"] = F.interpolate(out["label"][:, None].float(), size=(CFG.unet_size, CFG.unet_size), mode="nearest")[:, 0].long()
        return out


class Client:
    def __init__(self, name: str, man: pd.DataFrame, root: Path, raw=False):
        self.name = name; self.labelled = [CID[n] for n in CLIENT_LABELS[name]]; sub = man[man.client == name]
        self.split_rows = {s: sub[sub.split == s].row.tolist() for s in ("train", "cal", "test")}; self.video = dict(zip(sub.row, sub.video)); self.pah = dict(zip(sub.row, sub.pah)); self.fid = dict(zip(sub.row, sub.id))
        tr = sub[sub.split == "train"]; self.train_videos = {v: g.row.tolist() for v, g in tr.groupby("video")}
        self.store = FrameStore(sub.row.to_numpy(), root, raw)
    @property
    def n_train(self): return len(self.split_rows["train"])
    @property
    def n_train_videos(self): return len(self.train_videos)


def make_model(arm: dict, meta: dict) -> nn.Module:
    if arm.get("model") == "unet": return UNet2D(32)
    if arm.get("model") == "student": return UNet2D(16)
    m = meta[arm["encoder"]]; return FeatDecoder(m["c8"], m["c16"], norm=arm.get("norm", "bn"))


def is_raw(arm): return arm.get("model") in ("unet", "student")


def forward(model, batch, arm):
    return model(batch["frame"]) if is_raw(arm) else model(batch["s8"], batch["s16"], batch["label"].shape[-2:])


def task_loss(arm, logits, label, labelled):
    return (marginal_ce if arm["loss"] == "marginal" else bg_ce)(logits, label, labelled)


def kd_loss(logits, teacher_logits):
    return F.kl_div(logits.float().log_softmax(1), teacher_logits.float().softmax(1), reduction="batchmean") / (logits.shape[-1] * logits.shape[-2])


def teacher_logits(arm, client, batch):
    """Soft targets for the federated student: the federated teacher's client model on the same cached features, resized to the student grid."""
    t = arm["teacher"][client.name]; tb = {"s8": batch["s8"], "s16": batch["s16"], "label": batch["label_full"]}
    with torch.no_grad(): return F.interpolate(forward(t, tb, arm["teacher_arm"]).float(), size=batch["label"].shape[-2:], mode="bilinear", align_corners=False)


# ---------------- video-level DP-SGD (C3) --------------------------------------------------------------------------------
def dp_local_train(model, global_model, client, arm, state, rnd, rng) -> dict:
    """One sample = one VIDEO (K frames, loss summed over the frames), per-video gradients via torch.func.vmap, per-video clipping to
    CFG.dp_clip, Gaussian noise σ·C, Poisson sampling over videos, δ = 1/N_videos, Opacus RDP accountant per client across all rounds."""
    from torch.func import functional_call, vmap, grad
    from opacus.accountants import RDPAccountant
    from opacus.accountants.utils import get_noise_multiplier
    vids = list(client.train_videos); Nv = len(vids); K = CFG.dp_frames_per_video[client.name]; V = min(CFG.dp_videos_per_step, Nv); q = V / Nv
    steps_per_epoch = math.ceil(Nv / V); total = CFG.rounds * CFG.local_epochs * steps_per_epoch; delta = 1.0 / Nv; t0 = time.time()
    if client.name not in state["dp"]:
        sigma = get_noise_multiplier(target_epsilon=arm["dp"]["eps"], target_delta=delta, sample_rate=q, steps=total, accountant="rdp")
        state["dp"][client.name] = dict(sigma=float(sigma), q=q, delta=delta, unit="video", frames_per_video=K, clip_norm=CFG.dp_clip, steps_total=total, n_videos=Nv, accountant="RDP (Opacus)", accountant_obj=RDPAccountant(), steps=0)
    dps = state["dp"][client.name]; sigma = dps["sigma"]; acc = dps["accountant_obj"]
    model.train(); opt = torch.optim.Adam(model.parameters(), lr=CFG.lr); params = dict(model.named_parameters()); buffers = dict(model.named_buffers())
    g_params = {k: v.detach().clone() for k, v in global_model.named_parameters()} if global_model is not None else None; lab_ids = client.labelled
    def video_loss(p, b, s8, s16, lab, lg):
        logits = functional_call(model, (p, b), (s8, s16, lab.shape[-2:])); L = task_loss(arm, logits, lab, lab_ids) * K
        if arm.get("condist") and rnd > 0: L = L + CFG.condist_weight * min(1.0, rnd / max(1, 0.2 * CFG.rounds)) * condist_loss(logits, lg, lab, lab_ids) * K
        return L                                                 # FedProx's proximal term is added once per step below, outside vmap
    per_video = vmap(grad(video_loss), in_dims=(None, None, 0, 0, 0, 0), randomness="different"); n_steps, tot = 0, 0.0
    for ep in range(CFG.local_epochs):
        for s in range(steps_per_epoch):
            sel = np.nonzero(rng.random(Nv) < q)[0]
            if len(sel) == 0:
                acc.step(noise_multiplier=sigma, sample_rate=q); continue
            rows = [r for i in sel for r in (rng.choice(client.train_videos[vids[i]], K, replace=len(client.train_videos[vids[i]]) < K)).tolist()]
            batch = client.store.batch(rows); S = len(sel)
            lg = forward(global_model, batch, arm).detach() if (arm.get("condist") and rnd > 0 and global_model is not None) else torch.zeros_like(batch["label"], dtype=torch.float32)[:, None].expand(-1, N_CLS, -1, -1).contiguous()
            shp = lambda t: t.view(S, K, *t.shape[1:])
            grads = per_video(params, buffers, shp(batch["s8"]), shp(batch["s16"]), shp(batch["label"]), shp(lg))
            norms = torch.sqrt(sum((g.flatten(1) ** 2).sum(1) for g in grads.values())); factor = torch.clamp(CFG.dp_clip / (norms + 1e-6), max=1.0)
            for k, p in params.items():
                g = grads[k]; clipped = (g * factor.view(-1, *[1] * (g.dim() - 1))).sum(0)
                p.grad = (clipped + sigma * CFG.dp_clip * torch.randn_like(clipped)) / (q * Nv)
                if arm["algo"] == "fedprox" and g_params is not None: p.grad.add_(CFG.fedprox_mu * (p.detach() - g_params[k]))   # ∇ of μ/2‖w−w_g‖², public quantity, added once
            opt.step(); opt.zero_grad(); acc.step(noise_multiplier=sigma, sample_rate=q); n_steps += 1; dps["steps"] += 1
    return dict(steps=n_steps, loss=float("nan"), sec=time.time() - t0)


def local_train(model: nn.Module, global_model: Optional[nn.Module], client: Client, arm: dict, state: dict, rnd: int, rng: np.random.Generator) -> dict:
    """E local epochs on one client. Algorithms are selected by arm['algo'] and the loss by arm['loss'] / arm['condist'] / arm['teacher'] (KD)."""
    if arm.get("dp"): return dp_local_train(model, global_model, client, arm, state, rnd, rng)
    model.train(); rows_all = client.split_rows["train"]; N = len(rows_all); bs = min(CFG.batch, N); steps_per_epoch = math.ceil(N / bs); t0 = time.time()
    opt = torch.optim.SGD(model.parameters(), lr=CFG.lr * 20, momentum=0.0) if arm["algo"] == "scaffold" else torch.optim.Adam(model.parameters(), lr=CFG.lr)
    g_params = {k: v.detach().clone() for k, v in global_model.state_dict().items()} if global_model is not None else None
    c_glob, c_loc = state.get("c"), state.get("c_i", {}).get(client.name); n_steps, tot = 0, 0.0
    for ep in range(CFG.local_epochs):
        order = rng.permutation(N)
        for s in range(steps_per_epoch):
            idx = order[s * bs:(s + 1) * bs]; batch = client.store.batch([rows_all[i] for i in idx]); logits = forward(model, batch, arm)
            loss = task_loss(arm, logits, batch["label"], client.labelled)
            if arm.get("teacher") is not None: loss = loss + kd_loss(logits, teacher_logits(arm, client, batch))
            if global_model is not None and rnd > 0 and (arm.get("condist") or arm["algo"] == "fedconsist"):
                with torch.no_grad(): lg = forward(global_model, batch, arm)
                if arm.get("condist"): loss = loss + CFG.condist_weight * min(1.0, rnd / max(1, 0.2 * CFG.rounds)) * condist_loss(logits, lg, batch["label"], client.labelled)
                if arm["algo"] == "fedconsist":
                    pb = {k: (v + 0.1 * v.std() * torch.randn_like(v) if k in ("s8", "s16", "frame") else v) for k, v in batch.items()}
                    loss = loss + CFG.consist_weight * consistency_loss(forward(model, pb, arm), lg)
            if arm["algo"] == "fedprox" and g_params is not None:
                loss = loss + 0.5 * CFG.fedprox_mu * sum(((p - g_params[k]) ** 2).sum() for k, p in model.named_parameters())
            opt.zero_grad(); loss.backward()
            if arm["algo"] == "scaffold" and c_glob is not None:
                for k, p in model.named_parameters():
                    if p.grad is not None: p.grad.add_(c_glob[k] - c_loc[k])
            opt.step(); n_steps += 1; tot += float(loss)
    if arm["algo"] == "scaffold" and c_glob is not None and n_steps > 0:      # option II control-variate update
        lr = opt.param_groups[0]["lr"]; new_c = {k: c_loc[k] - c_glob[k] + (g_params[k] - p.detach()) / (n_steps * lr) for k, p in model.named_parameters()}
        state["dc"][client.name] = {k: new_c[k] - c_loc[k] for k in new_c}; state["c_i"][client.name] = new_c
    return dict(steps=n_steps, loss=tot / max(1, n_steps), sec=time.time() - t0)


def aggregate(global_model: nn.Module, client_models: Dict[str, nn.Module], clients: Dict[str, Client], keys: List[str], arm: dict, secure: bool, rng) -> bool:
    """Weighted FedAvg over the transmitted keys (FedPSL-style: head rows per class only among clients that label that class).
    With `secure`, the client vectors pass through a simulated pairwise-mask secure aggregation; the result must equal the plain sum."""
    names = list(client_models); w = np.array([clients[n].n_train for n in names], float); w /= w.sum(); sds = {n: client_models[n].state_dict() for n in names}; new = {}
    ok = True
    for k in keys:
        ref = sds[names[0]][k]
        if not ref.dtype.is_floating_point: new[k] = ref.clone(); continue
        if arm["algo"] == "fedpsl" and k.startswith("head."):
            out = torch.zeros_like(ref)
            for c in range(N_CLS):
                elig = [i for i, n in enumerate(names) if c == 0 or c in clients[n].labelled] or list(range(len(names))); ww = w[elig] / w[elig].sum()
                out[c] = sum(ww[j] * sds[names[i]][k][c] for j, i in enumerate(elig))
            new[k] = out; continue
        vecs = [w[i] * sds[n][k].float() for i, n in enumerate(names)]; plain = sum(vecs)
        if secure:
            masked = []; G = torch.Generator().manual_seed(int(rng.integers(0, 2 ** 31)))
            masks = {(i, j): torch.randn(ref.shape, generator=G).to(ref.device) for i in range(len(names)) for j in range(i + 1, len(names))}
            for i in range(len(names)):
                v = vecs[i].clone()
                for j in range(len(names)):
                    if i < j: v += masks[(i, j)]
                    elif j < i: v -= masks[(j, i)]
                masked.append(v)
            agg = sum(masked); ok &= bool(torch.allclose(agg, plain, atol=1e-4, rtol=1e-4)); new[k] = agg.to(ref.dtype)
        else: new[k] = plain.to(ref.dtype)
    sd = global_model.state_dict(); sd.update(new); global_model.load_state_dict(sd)
    return ok


def sync_from_global(client_model, global_model, keys):
    sd = client_model.state_dict(); g = global_model.state_dict()
    for k in keys: sd[k] = g[k].clone()
    client_model.load_state_dict(sd)


@torch.no_grad()
def evaluate_client(model: nn.Module, client: Client, arm: dict, split="test") -> pd.DataFrame:
    """Per-(frame, structure) Dice and HD95 for the structures the client labels; LV/RV areas kept for the RV/LV-ratio endpoint."""
    model.eval(); rows = []; rws = client.split_rows[split]
    for b0 in range(0, len(rws), 32):
        bid = rws[b0:b0 + 32]; batch = client.store.batch(bid)
        with autocast_ctx(): pred = forward(model, batch, arm).argmax(1).cpu().numpy()
        lab = batch["label"].cpu().numpy()
        for i, r in enumerate(bid):
            a_lv, a_rv = float((pred[i] == CID["LV"]).mean()), float((pred[i] == CID["RV"]).mean())
            for c in client.labelled:
                p, g = pred[i] == c, lab[i] == c
                rows.append(dict(client=client.name, id=client.fid[r], video=client.video[r], pah=client.pah[r], structure=CLASSES[c], dice=dice_binary(p, g), hd=hausdorff(p, g), area_lv=a_lv, area_rv=a_rv))
    return pd.DataFrame(rows)


def summarise(perframe: pd.DataFrame) -> dict:
    """Three labelled aggregates (D6): global_dice = mean over client×structure pairs; global_dice_client_mean = mean over clients of the
    client-mean Dice; global_dice_structure_pooled = mean over structures of the Dice pooled across the clients that label them
    (pooled_dice_<structure>). Plus per-client/structure Dice & HD95, worst client and the held-out numbers."""
    cs = perframe.groupby(["client", "structure"]).agg(dice=("dice", "mean"), hd=("hd", "mean")); out = {}
    for (c, s), r in cs.iterrows(): out[f"dice_{c}_{s}"] = float(r.dice); out[f"hd95_{c}_{s}"] = float(r.hd)
    cm = cs.groupby("client").dice.mean(); out.update({f"client_dice_{c}": float(v) for c, v in cm.items()})
    pooled = perframe.groupby("structure").dice.mean(); out.update({f"pooled_dice_{s}": float(v) for s, v in pooled.items()})
    out["global_dice"] = float(cs.dice.mean()); out["global_dice_client_mean"] = float(cm.mean()); out["global_dice_structure_pooled"] = float(pooled.mean()); out["worst_client_dice"] = float(cm.min())
    for c, structs in HELDOUT_STRUCTURES.items():
        for s in structs: out[f"heldout_{c}_{s}"] = out.get(f"dice_{c}_{s}", np.nan)
    return out


def run_federated(arm: dict, clients: Dict[str, Client], meta: dict, tag: str) -> dict:
    """One arm × seed. arm keys: name, algo ∈ {local, central, fedavg, fedprox, scaffold, fedinit, fedconsist, fedpsl}, loss ∈ {bg, marginal},
    condist, fedbn, encoder, norm, model ('feat'|'unet'|'student'), dp ({eps} or None), secure_agg, teacher (+teacher_arm) for KD, seed."""
    seed_everything(arm["seed"]); rng = np.random.default_rng(arm["seed"]); names = list(clients)
    global_model = make_model(arm, meta).to(DEVICE); client_models = {n: copy.deepcopy(global_model) for n in names}
    keys = transmitted_keys(global_model, arm.get("fedbn", False)); n_tx = sum(global_model.state_dict()[k].numel() for k in keys)
    state = dict(dp={}, time={n: 0.0 for n in names}, dc={}, c=None, c_i={}, prev_local={})
    if arm["algo"] == "scaffold":
        state["c"] = {k: torch.zeros_like(p) for k, p in global_model.named_parameters()}; state["c_i"] = {n: {k: torch.zeros_like(p) for k, p in global_model.named_parameters()} for n in names}
    history, secure_ok = [], True
    def evaluate_all(rnd):
        pf = pd.concat([evaluate_client(client_models[n], clients[n], arm) for n in names], ignore_index=True); s = summarise(pf); history.append(dict(round=rnd, **s)); return pf
    if arm["algo"] == "local":                                   # lower reference: each client alone, equal local compute
        for rnd in range(CFG.rounds):
            for n in names: st = local_train(client_models[n], None, clients[n], arm, state, rnd, rng); state["time"][n] += st["sec"]
            if (rnd + 1) % CFG.eval_every == 0 or rnd == CFG.rounds - 1: pf = evaluate_all(rnd + 1)
        n_tx = 0
    elif arm["algo"] == "central":                               # pooled data, one model, per-client label sets for the loss
        opt = torch.optim.Adam(global_model.parameters(), lr=CFG.lr)
        for rnd in range(CFG.rounds):
            t0 = time.time(); global_model.train()
            for ep in range(CFG.local_epochs):
                queues = {n: rng.permutation(clients[n].n_train).tolist() for n in names}
                while any(queues.values()):
                    for n in names:
                        if not queues[n]: continue
                        idx, queues[n] = queues[n][:CFG.batch], queues[n][CFG.batch:]; batch = clients[n].store.batch([clients[n].split_rows["train"][i] for i in idx]); logits = forward(global_model, batch, arm)
                        loss = task_loss(arm, logits, batch["label"], clients[n].labelled)
                        if arm.get("teacher") is not None: loss = loss + kd_loss(logits, teacher_logits(arm, clients[n], batch))
                        opt.zero_grad(); loss.backward(); opt.step()
            for n in names: state["time"][n] += (time.time() - t0) / len(names); sync_from_global(client_models[n], global_model, list(global_model.state_dict()))
            if (rnd + 1) % CFG.eval_every == 0 or rnd == CFG.rounds - 1: pf = evaluate_all(rnd + 1)
        n_tx = 0
    else:
        for rnd in range(CFG.rounds):
            for n in names:
                sync_from_global(client_models[n], global_model, keys)
                if arm["algo"] == "fedinit" and n in state["prev_local"]:      # FedInit relaxed initialisation: w_i ← w_g + β (w_g − w_i^{last local}) (C1)
                    sd = client_models[n].state_dict(); prev = state["prev_local"][n]
                    for k in keys:
                        if sd[k].dtype.is_floating_point: sd[k] = sd[k] + CFG.fedinit_beta * (sd[k] - prev[k])
                    client_models[n].load_state_dict(sd)
                st = local_train(client_models[n], global_model, clients[n], arm, state, rnd, rng); state["time"][n] += st["sec"]
                if arm["algo"] == "fedinit": state["prev_local"][n] = {k: v.detach().clone() for k, v in client_models[n].state_dict().items() if k in keys}   # last LOCAL state, before aggregation
            secure_ok &= aggregate(global_model, client_models, clients, keys, arm, arm.get("secure_agg", False), rng)
            if arm["algo"] == "scaffold":
                for k in state["c"]: state["c"][k] = state["c"][k] + sum(state["dc"][n][k] for n in names) / len(names)
            for n in names: sync_from_global(client_models[n], global_model, keys)
            if (rnd + 1) % CFG.eval_every == 0 or rnd == CFG.rounds - 1: pf = evaluate_all(rnd + 1)
    final = history[-1]; eps = {n: float(d["accountant_obj"].get_epsilon(d["delta"])) for n, d in state["dp"].items()}
    return dict(history=history, final=final, perframe=pf, n_tx=n_tx, mb_per_round=2 * len(names) * n_tx * 4 / 2 ** 20, secure_agg_ok=secure_ok,
                gpu_min_per_client={n: t / 60 for n, t in state["time"].items()}, eps_spent=eps, dp_params={n: {k: v for k, v in d.items() if k != "accountant_obj"} for n, d in state["dp"].items()},
                global_state=global_model.state_dict(), client_states={n: m.state_dict() for n, m in client_models.items()})


def clients_for(encoder: str, raw=False) -> Tuple[Dict[str, Client], dict]:
    d, man = load_manifest(encoder); meta = json.loads((d / "meta.json").read_text())
    return {n: Client(n, man, d, raw=raw) for n in CLIENT_LABELS}, meta


def run_and_log(arm: dict, out: Path, log: RunLog, manifest: RunManifest, clients=None, meta=None, save_model=True) -> Optional[dict]:
    rid = f"{arm['name']}_s{arm['seed']}"
    if manifest.done(rid): return None
    if clients is None: clients, meta = clients_for(arm["encoder"], raw=is_raw(arm))
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    with Timer() as tm: res = run_federated(arm, clients, meta, rid)
    res["perframe"].to_csv(out / f"perframe_{rid}.csv", index=False); pd.DataFrame(res["history"]).to_csv(out / f"history_{rid}.csv", index=False)
    if save_model: torch.save({"global": res["global_state"], "clients": res["client_states"], "arm": {k: v for k, v in arm.items() if k not in ("teacher", "teacher_arm")}}, out / f"model_{rid}.pt")
    row = dict(stage=arm.get("stage"), arm=arm["name"], algo=arm["algo"], loss=arm["loss"], condist=arm.get("condist", False), fedbn=arm.get("fedbn", False), encoder=arm["encoder"], norm=arm.get("norm", "bn"),
               model=arm.get("model", "feat"), optimizer=("sgd(lr×20, no momentum)" if arm["algo"] == "scaffold" else "adam"), dp_eps_target=(arm["dp"]["eps"] if arm.get("dp") else np.nan), dp_unit=(CFG.dp_unit if arm.get("dp") else ""),
               seed=arm["seed"], rounds=CFG.rounds, local_epochs=CFG.local_epochs, **res["final"], n_tx_params=res["n_tx"], mb_per_round=res["mb_per_round"], secure_agg_ok=res["secure_agg_ok"],
               eps_spent=json.dumps(res["eps_spent"]), dp_params=json.dumps(res["dp_params"]), gpu_min_per_client=json.dumps(res["gpu_min_per_client"]), sec=tm.dt, peak_mem_mb=peak_mem_mb(), params_model=count_params(make_model(arm, meta)))
    log.append(rid, row); manifest.mark(rid)
    print(f"[{arm['name']:34s} s{arm['seed']}] global Dice {res['final']['global_dice']:.3f} | worst client {res['final']['worst_client_dice']:.3f} | "
          f"CardiacNet RV {res['final'].get('heldout_CARDIACNET_RV', np.nan):.3f} RA {res['final'].get('heldout_CARDIACNET_RA', np.nan):.3f} | tx {res['n_tx']/1e6:.2f} M params, {res['mb_per_round']:.1f} MB/round"
          + (f" | ε spent {res['eps_spent']}" if res["eps_spent"] else "") + f" ({tm.dt:.0f}s)")
    return row

In [7]:
# ---------------------------------------------------------------- Cell 7: Stage 11 — local-only, centralised and standard federated baselines (S11)
def arm(name, algo, loss="bg", stage=11, encoder=None, **kw):
    return dict(name=name, algo=algo, loss=loss, stage=stage, encoder=encoder or CFG.encoders[0], **kw)


BASELINE_ARMS = [arm("local_only", "local"), arm("central_bg", "central"), arm("central_marginal", "central", "marginal"),
                 arm("fedavg_bg", "fedavg"), arm("fedprox_bg", "fedprox"), arm("scaffold_bg", "scaffold"), arm("fedinit_bg", "fedinit"), arm("fedconsist_bg", "fedconsist"), arm("fedpsl_bg", "fedpsl")]


def stage11_baselines():
    require_inputs(11)
    out = stage_dir(CFG.ds_base, create=True); log = RunLog(out); manifest = RunManifest(out / "runs_manifest.json"); clients, meta = clients_for(CFG.encoders[0])
    for a in BASELINE_ARMS:
        for seed in CFG.seeds:
            run_and_log({**a, "seed": seed}, out, log, manifest, clients, meta)
    write_version(CFG.ds_base, dict(arms=[a["name"] for a in BASELINE_ARMS], seeds=CFG.seeds, scaffold_optimizer="SGD lr×20 without momentum (control variates); every other arm Adam"))
    df = log.read()
    fa, fi = df[df.arm == "fedavg_bg"].sort_values("seed").global_dice.to_numpy(), df[df.arm == "fedinit_bg"].sort_values("seed").global_dice.to_numpy()
    assert not np.allclose(fa, fi), "FedInit reproduced FedAvg exactly — the relaxed initialisation is a no-op (C1)"
    print(f"[stage 11] FedInit ≠ FedAvg check passed (global Dice {fa.round(3)} vs {fi.round(3)})"); print(df.groupby("arm")[["global_dice", "worst_client_dice", "heldout_CARDIACNET_RV", "heldout_CARDIACNET_RA", "heldout_CAMUS_LV", "heldout_CAMUS_LA"]].mean().round(3))


if 11 in CFG.stages:
    stage11_baselines()

[stage 11] depends on: ['fed-features-dinov3-v1']
   fed-features-dinov3-v1 -> k2_work/fed-features-dinov3-v1  (2026-09-29T13:37:27)


[local_only                         s0] global Dice 0.301 | worst client 0.227 | CardiacNet RV 0.548 RA 0.000 | tx 0.00 M params, 0.0 MB/round (8s)


[local_only                         s1] global Dice 0.337 | worst client 0.261 | CardiacNet RV 0.564 RA 0.000 | tx 0.00 M params, 0.0 MB/round (6s)


[central_bg                         s0] global Dice 0.777 | worst client 0.547 | CardiacNet RV 0.797 RA 0.838 | tx 0.00 M params, 0.0 MB/round (7s)


[central_bg                         s1] global Dice 0.009 | worst client 0.000 | CardiacNet RV 0.063 RA 0.000 | tx 0.00 M params, 0.0 MB/round (6s)


...

[fedinit_bg                         s1] global Dice 0.278 | worst client 0.000 | CardiacNet RV 0.000 RA 0.011 | tx 0.58 M params, 13.2 MB/round (6s)


[fedconsist_bg                      s0] global Dice 0.261 | worst client 0.042 | CardiacNet RV 0.000 RA 0.000 | tx 0.58 M params, 13.2 MB/round (13s)


[fedconsist_bg                      s1] global Dice 0.335 | worst client 0.001 | CardiacNet RV 0.000 RA 0.162 | tx 0.58 M params, 13.2 MB/round (14s)


[fedpsl_bg                          s0] global Dice 0.236 | worst client 0.085 | CardiacNet RV 0.000 RA 0.000 | tx 0.58 M params, 13.2 MB/round (7s)


[fedpsl_bg                          s1] global Dice 0.351 | worst client 0.000 | CardiacNet RV 0.000 RA 0.000 | tx 0.58 M params, 13.2 MB/round (7s)
[stage 11] FedInit ≠ FedAvg check passed (global Dice [0.196 0.271] vs [0.299 0.278])
                  global_dice  worst_client_dice  heldout_CARDIACNET_RV  \
arm                                                                       
central_bg              0.393              0.274                  0.430   
central_marginal        0.762              0.665                  0.759   
fedavg_bg               0.234              0.001                  0.000   
fedconsist_bg           0.298              0.022                  0.000   
fedinit_bg              0.288              0.011                  0.000   
fedprox_bg              0.139              0.000                  0.000   
fedpsl_bg               0.294              0.042                  0.000   
local_only              0.319              0.244                  0.556   
scaffold_bg    

In [8]:
# ---------------------------------------------------------------- Cell 8: Stage 12 — marginal loss, conditional distillation, FedBN, encoder swap, full-model U-Net reference (S12)
def method_arms() -> List[dict]:
    e0 = CFG.encoders[0]
    arms = [arm(f"fedavg_marginal", "fedavg", "marginal", 12), arm("fedavg_marginal_condist", "fedavg", "marginal", 12, condist=True),
            arm("fedavg_marginal_condist_fedbn", "fedavg", "marginal", 12, condist=True, fedbn=True)]
    for enc in CFG.encoders[1:]:                                   # frozen-encoder swap: does echo-specific pretraining matter once frozen?
        arms += [arm(f"fedavg_bg__{enc}", "fedavg", "bg", 12, encoder=enc), arm(f"fedavg_marginal_condist_fedbn__{enc}", "fedavg", "marginal", 12, encoder=enc, condist=True, fedbn=True)]
    arms.append(arm("unet_fullmodel_fedavg_bg", "fedavg", "bg", 12, model="unet"))   # communication reference on raw 112² frames
    return arms


def stage12_method():
    require_inputs(12)
    out = stage_dir(CFG.ds_method, create=True); log = RunLog(out); manifest = RunManifest(out / "runs_manifest.json"); cache = {}
    for a in method_arms():
        key = (a["encoder"], a.get("model", "feat"))
        if key not in cache: cache[key] = clients_for(a["encoder"], raw=is_raw(a))
        for seed in CFG.seeds:
            run_and_log({**a, "seed": seed}, out, log, manifest, *cache[key])
    write_version(CFG.ds_method, dict(arms=[a["name"] for a in method_arms()], seeds=CFG.seeds))
    df = pd.concat([RunLog(stage_dir(CFG.ds_base)).read(), log.read()], ignore_index=True)
    print(df.groupby("arm")[["global_dice", "global_dice_client_mean", "global_dice_structure_pooled", "worst_client_dice", "heldout_CARDIACNET_RV", "heldout_CARDIACNET_RA", "n_tx_params", "mb_per_round"]].mean().round(3).sort_values("global_dice", ascending=False))


if 12 in CFG.stages:
    stage12_method()

[stage 12] depends on: ['fed-features-dinov3-v1', 'fed-features-panecho-v1', 'fed-baselines-v1']
   fed-features-dinov3-v1 -> k2_work/fed-features-dinov3-v1  (2026-09-29T13:37:27)
   fed-features-panecho-v1 -> k2_work/fed-features-panecho-v1  (2026-09-29T13:37:27)
   fed-baselines-v1     -> k2_work/fed-baselines-v1  (2026-09-29T13:39:42)


[fedavg_marginal                    s0] global Dice 0.427 | worst client 0.297 | CardiacNet RV 0.669 RA 0.069 | tx 0.58 M params, 13.2 MB/round (7s)


[fedavg_marginal                    s1] global Dice 0.520 | worst client 0.365 | CardiacNet RV 0.581 RA 0.260 | tx 0.58 M params, 13.2 MB/round (7s)


[fedavg_marginal_condist            s0] global Dice 0.599 | worst client 0.379 | CardiacNet RV 0.794 RA 0.757 | tx 0.58 M params, 13.2 MB/round (9s)


[fedavg_marginal_condist            s1] global Dice 0.623 | worst client 0.487 | CardiacNet RV 0.820 RA 0.697 | tx 0.58 M params, 13.2 MB/round (10s)


...

[fedavg_bg__panecho                 s1] global Dice 0.271 | worst client 0.000 | CardiacNet RV 0.000 RA 0.000 | tx 0.58 M params, 13.2 MB/round (6s)


[fedavg_marginal_condist_fedbn__panecho s0] global Dice 0.555 | worst client 0.121 | CardiacNet RV 0.815 RA 0.716 | tx 0.58 M params, 13.2 MB/round (8s)


[fedavg_marginal_condist_fedbn__panecho s1] global Dice 0.248 | worst client 0.127 | CardiacNet RV 0.701 RA 0.000 | tx 0.58 M params, 13.2 MB/round (8s)


[unet_fullmodel_fedavg_bg           s0] global Dice 0.663 | worst client 0.001 | CardiacNet RV 0.928 RA 0.587 | tx 7.77 M params, 177.9 MB/round (75s)


[unet_fullmodel_fedavg_bg           s1] global Dice 0.602 | worst client 0.007 | CardiacNet RV 0.044 RA 0.917 | tx 7.77 M params, 177.9 MB/round (77s)
                                        global_dice  global_dice_client_mean  \
arm                                                                            
central_marginal                              0.762                    0.737   
unet_fullmodel_fedavg_bg                      0.633                    0.476   
fedavg_marginal_condist                       0.611                    0.569   
fedavg_marginal                               0.474                    0.432   
fedavg_marginal_condist_fedbn                 0.401                    0.344   
fedavg_marginal_condist_fedbn__panecho        0.401                    0.344   
central_bg                                    0.393                    0.361   
local_only                                    0.319                    0.371   
fedconsist_bg                                 0.2

In [9]:
# ---------------------------------------------------------------- Cell 9: Stage 13 — DP-SGD utility–ε curve, secure aggregation, membership-inference audit, conformal margins (S13)
def best_feature_arms(n=2) -> List[dict]:
    """The n best non-private feature-space federated configurations by mean global Dice over seeds (stages 11–12)."""
    df = pd.concat([RunLog(stage_dir(CFG.ds_base)).read(), RunLog(stage_dir(CFG.ds_method)).read()], ignore_index=True)
    df = df[~df.algo.isin(["local", "central"]) & (df.model == "feat") & (df.encoder == CFG.encoders[0])]
    best = df.groupby("arm").global_dice.mean().sort_values(ascending=False).index[:n]
    return [dict(name=r.arm, algo=r.algo, loss=r.loss, condist=bool(r.condist), fedbn=bool(r.fedbn), encoder=r.encoder) for r in df.drop_duplicates("arm").set_index("arm").loc[best].reset_index().itertuples()]


def load_arm_model(path: Path, meta: dict):
    ck = torch.load(path, map_location=DEVICE, weights_only=False); a = ck["arm"]; models = {}
    for n, sd in ck["clients"].items():
        m = make_model(a, meta).to(DEVICE); m.load_state_dict(sd); models[n] = m.eval()
    return a, models


@torch.no_grad()
def frame_features(model, client, arm, rws) -> pd.DataFrame:
    """Per-frame attack features: task loss, mean max-probability, mean entropy."""
    model.eval(); rows = []
    for b0 in range(0, len(rws), 32):
        bid = rws[b0:b0 + 32]; batch = client.store.batch(bid); logits = forward(model, batch, arm).float(); p = logits.softmax(1)
        for i, r in enumerate(bid):
            l = task_loss(arm, logits[i:i + 1], batch["label"][i:i + 1], client.labelled)
            rows.append(dict(id=client.fid[r], video=client.video[r], loss=float(l), maxp=float(p[i].max(0)[0].mean()), entropy=float(-(p[i] * p[i].clamp_min(1e-8).log()).sum(0).mean())))
    return pd.DataFrame(rows)


def mia_audit(model_path: Path, clients: Dict[str, Client], meta: dict, seed=0) -> dict:
    """Loss-threshold (Yeom et al.) and shadow-model attacks on the aggregated model, per client: members = training frames,
    non-members = test frames (balanced). The shadow decoder is trained on half of the client's cal+test videos (disjoint from the
    target's training data); an attack classifier on [loss, max-prob, entropy] is transferred to the target."""
    from sklearn.linear_model import LogisticRegression
    a, models = load_arm_model(model_path, meta); rng = np.random.default_rng(seed); out = {}
    for n, c in clients.items():
        te = c.split_rows["test"]; tr = list(rng.choice(c.split_rows["train"], size=min(len(te), c.n_train), replace=False))
        f_in, f_out = frame_features(models[n], c, a, tr), frame_features(models[n], c, a, te)
        y = np.r_[np.ones(len(f_in)), np.zeros(len(f_out))]; X = pd.concat([f_in, f_out], ignore_index=True)
        out[f"{n}_loss_threshold_auc"] = float(roc_auc_score(y, -X.loss))
        pool = c.split_rows["cal"] + te; vids = sorted({c.video[i] for i in pool}); rng.shuffle(vids); half = set(vids[:len(vids) // 2])
        A = [i for i in pool if c.video[i] in half]; B = [i for i in pool if c.video[i] not in half]
        shadow = make_model(a, meta).to(DEVICE); opt = torch.optim.Adam(shadow.parameters(), lr=CFG.lr); shadow.train()
        for ep in range(max(2, CFG.local_epochs * 4)):
            for b0 in range(0, len(A), CFG.batch):
                batch = c.store.batch(A[b0:b0 + CFG.batch]); loss = task_loss(a, forward(shadow, batch, a), batch["label"], c.labelled); opt.zero_grad(); loss.backward(); opt.step()
        s_in, s_out = frame_features(shadow, c, a, A), frame_features(shadow, c, a, B)
        Xs = pd.concat([s_in, s_out], ignore_index=True)[["loss", "maxp", "entropy"]].to_numpy(); ys = np.r_[np.ones(len(s_in)), np.zeros(len(s_out))]
        clf = LogisticRegression(max_iter=1000).fit(Xs, ys); out[f"{n}_shadow_auc"] = float(roc_auc_score(y, clf.predict_proba(X[["loss", "maxp", "entropy"]].to_numpy())[:, 1]))
    out["mean_loss_threshold_auc"] = float(np.mean([v for k, v in out.items() if k.endswith("loss_threshold_auc")])); out["mean_shadow_auc"] = float(np.mean([v for k, v in out.items() if k.endswith("shadow_auc")]))
    return out


@torch.no_grad()
def conformal_margins(model_path: Path, clients: Dict[str, Client], meta: dict, alpha: float) -> pd.DataFrame:
    """Per-client morphological conformal margins (client calibration sets, never pooled): the smallest dilation radius whose dilated
    prediction contains the truth (outer) and the smallest erosion radius whose eroded prediction lies inside it (inner) are the
    nonconformity scores; their finite-sample (1−α) quantiles give margins whose coverage is then measured on the client's test set."""
    a, models = load_arm_model(model_path, meta); rows = []; R = CFG.conformal_max_margin
    def margins(pred, gt):
        r_out = next((r for r in range(R + 1) if np.all(gt <= (ndimage.binary_dilation(pred, iterations=r) if r else pred))), R + 1)
        r_in = next((r for r in range(R + 1) if np.all((ndimage.binary_erosion(pred, iterations=r) if r else pred) <= gt)), R + 1)
        return r_out, r_in
    def scores(model, c, split):
        out = {s: [] for s in c.labelled}; rws = c.split_rows[split]
        for b0 in range(0, len(rws), 32):
            batch = c.store.batch(rws[b0:b0 + 32]); pred = forward(model, batch, a).argmax(1).cpu().numpy(); lab = batch["label"].cpu().numpy()
            for i in range(len(pred)):
                for s in c.labelled: out[s].append(margins(pred[i] == s, lab[i] == s))
        return {s: np.array(v) for s, v in out.items()}
    for n, c in clients.items():
        cal, te = scores(models[n], c, "cal"), scores(models[n], c, "test")
        for s in c.labelled:
            m = len(cal[s]); lvl = min(1.0, math.ceil((m + 1) * (1 - alpha)) / m)
            q_out, q_in = np.quantile(cal[s][:, 0], lvl, method="higher"), np.quantile(cal[s][:, 1], lvl, method="higher")
            cov = float(np.mean((te[s][:, 0] <= q_out) & (te[s][:, 1] <= q_in)))
            rows.append(dict(client=n, structure=CLASSES[s], n_cal=m, q_outer_px=float(q_out), q_inner_px=float(q_in), test_coverage=cov, target=1 - alpha, mean_outer_cal=float(cal[s][:, 0].mean()), mean_inner_cal=float(cal[s][:, 1].mean())))
    return pd.DataFrame(rows)


def stage13_privacy():
    require_inputs(13)
    out = stage_dir(CFG.ds_priv, create=True); log = RunLog(out); manifest = RunManifest(out / "runs_manifest.json"); clients, meta = clients_for(CFG.encoders[0])
    best = best_feature_arms(2); print("[stage 13] DP on the two best configurations:", [b["name"] for b in best])
    # --- utility–ε curve: GroupNorm reference (ε = ∞) and DP-SGD at ε ∈ {1,4,8}, simulated secure aggregation on every DP round -----
    for b in best:
        for eps in [None] + CFG.dp_eps:
            a = {**b, "stage": 13, "norm": "gn", "dp": ({"eps": eps} if eps else None), "secure_agg": bool(eps), "name": f"{b['name']}__gn_eps{eps if eps else 'inf'}"}
            for seed in CFG.seeds: run_and_log({**a, "seed": seed}, out, log, manifest, clients, meta)
    # --- membership-inference audit on every model of the curve (seed 0) and conformal margins for the best configuration --------------
    df = log.read(); audits, conf = [], []
    for _, r in df[df.seed == 0].iterrows():
        au = mia_audit(out / f"model_{r.run_id}.pt", clients, meta); audits.append(dict(arm=r.arm, eps_target=r.dp_eps_target, eps_spent=r.eps_spent, global_dice=r.global_dice, **au))
        print(f"[stage 13] MIA {r.arm}: loss-threshold AUROC {au['mean_loss_threshold_auc']:.3f} | shadow AUROC {au['mean_shadow_auc']:.3f}")
    pd.DataFrame(audits).to_csv(out / "membership_inference.csv", index=False)
    for tag in (f"{best[0]['name']}__gn_epsinf", f"{best[0]['name']}__gn_eps{CFG.dp_eps[-1]}"):
        cm = conformal_margins(out / f"model_{tag}_s0.pt", clients, meta, 1 - CFG.coverage); cm["arm"] = tag; conf.append(cm)
    conf = pd.concat(conf, ignore_index=True); conf.to_csv(out / "conformal_margins.csv", index=False); print(conf.round(3).to_string(index=False))
    dp_rows = df[df.dp_eps_target.notna() & (df.seed == 0)]
    write_version(CFG.ds_priv, dict(best=[b["name"] for b in best], eps_targets=CFG.dp_eps, clip_norm=CFG.dp_clip, accountant="RDP (Opacus RDPAccountant, composed over all rounds per client)",
                                    unit=f"{CFG.dp_unit}-level: one sample = one video with K frames (K = {CFG.dp_frames_per_video}), Poisson sampling over videos, δ = 1/N_videos(client)",
                                    per_arm={r.arm: dict(eps_spent=json.loads(r.eps_spent), params=json.loads(r.dp_params)) for _, r in dp_rows.iterrows()}))
    print(df.groupby(["arm"])[["global_dice", "worst_client_dice", "secure_agg_ok"]].mean().round(3))


if 13 in CFG.stages:
    stage13_privacy()

[stage 13] depends on: ['fed-features-dinov3-v1', 'fed-baselines-v1', 'fed-method-v1']
   fed-features-dinov3-v1 -> k2_work/fed-features-dinov3-v1  (2026-09-29T13:37:27)
   fed-baselines-v1     -> k2_work/fed-baselines-v1  (2026-09-29T13:39:42)
   fed-method-v1        -> k2_work/fed-method-v1  (2026-09-29T13:43:34)


[stage 13] DP on the two best configurations: ['fedavg_marginal_condist', 'fedavg_marginal']


[fedavg_marginal_condist__gn_epsinf s0] global Dice 0.585 | worst client 0.305 | CardiacNet RV 0.792 RA 0.687 | tx 0.58 M params, 13.2 MB/round (9s)


[fedavg_marginal_condist__gn_epsinf s1] global Dice 0.589 | worst client 0.401 | CardiacNet RV 0.766 RA 0.683 | tx 0.58 M params, 13.2 MB/round (9s)


[fedavg_marginal_condist__gn_eps1.0 s0] global Dice 0.026 | worst client 0.000 | CardiacNet RV 0.207 RA 0.000 | tx 0.58 M params, 13.2 MB/round | ε spent {'CAMUS': 0.9925964822612899, 'HMCQU': 0.9980915095728913, 'CARDIACNET': 0.997834114690056} (18s)


...

[stage 13] MIA fedavg_marginal_condist__gn_eps1.0: loss-threshold AUROC 0.587 | shadow AUROC 0.592


[stage 13] MIA fedavg_marginal_condist__gn_eps4.0: loss-threshold AUROC 0.616 | shadow AUROC 0.616


[stage 13] MIA fedavg_marginal_condist__gn_eps8.0: loss-threshold AUROC 0.631 | shadow AUROC 0.626


[stage 13] MIA fedavg_marginal_condist__gn_epsinf: loss-threshold AUROC 0.658 | shadow AUROC 0.658


    client structure  n_cal  q_outer_px  q_inner_px  test_coverage  target  mean_outer_cal  mean_inner_cal                                arm
     CAMUS        LV      8         7.0         7.0          1.000     0.9           4.500           5.000 fedavg_marginal_condist__gn_epsinf
     CAMUS       MYO      8         7.0         4.0          1.000     0.9           7.000           3.375 fedavg_marginal_condist__gn_epsinf
     CAMUS        LA      8         7.0         7.0          1.000     0.9           4.000           4.625 fedavg_marginal_condist__gn_epsinf
     HMCQU       MYO      8         7.0         4.0          1.000     0.9           6.875           3.375 fedavg_marginal_condist__gn_epsinf
CARDIACNET        LV     10         7.0         5.0          0.933     0.9           3.500           3.700 fedavg_marginal_condist__gn_epsinf
CARDIACNET        RV     10         6.0         7.0          0.867     0.9           2.200           5.200 fedavg_marginal_condist__gn_epsinf
CARDIA

In [10]:
# ---------------------------------------------------------------- Cell 10: Stage 14 — student distillation, RV/LV ratio vs RHC label, statistics, withdrawal conditions, figures (S14)
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt


def all_runs() -> pd.DataFrame:
    return pd.concat([RunLog(stage_dir(d)).read() for d in (CFG.ds_base, CFG.ds_method, CFG.ds_priv)], ignore_index=True)


def perframe(ds: str, arm: str) -> pd.DataFrame:
    """Per-frame Dice averaged over seeds for an arm (one row per frame × structure)."""
    fs = sorted(stage_dir(ds).glob(f"perframe_{arm}_s*.csv")); assert fs, f"no per-frame files for {arm} in {ds}"
    return pd.concat([pd.read_csv(f) for f in fs]).groupby(["client", "id", "video", "structure"], as_index=False).agg(dice=("dice", "mean"), hd=("hd", "mean"), pah=("pah", "first"), area_lv=("area_lv", "mean"), area_rv=("area_rv", "mean"))


def paired_video_bootstrap(A: pd.DataFrame, B: pd.DataFrame, n: int, seed=0) -> dict:
    """Cluster (video/patient) bootstrap of the mean per-frame Dice difference A − B on one client."""
    M = A.merge(B, on=["client", "id", "video", "structure"], suffixes=("_a", "_b")); d = (M.dice_a - M.dice_b).to_numpy(); gi = pd.factorize(M.video)[0]; G = gi.max() + 1
    rng = np.random.default_rng(seed); reps = []
    for _ in range(n):
        w = np.bincount(rng.integers(0, G, G), minlength=G)[gi]; reps.append((d * w).sum() / max(w.sum(), 1))
    return dict(delta=float(d.mean()), ci_lo=float(np.percentile(reps, 2.5)), ci_hi=float(np.percentile(reps, 97.5)), n_videos=int(G))


def permutation_p(A: pd.DataFrame, B: pd.DataFrame, n: int, seed=0) -> float:
    """Video-level sign-flip permutation test of the mean paired Dice difference A − B (arm identity swapped for all frames of a video)."""
    M = A.merge(B, on=["client", "id", "video", "structure"], suffixes=("_a", "_b")); d = (M.dice_a - M.dice_b).to_numpy(); gi = pd.factorize(M.video)[0]; G = gi.max() + 1
    obs = abs(d.mean()); rng = np.random.default_rng(seed); hits = 0
    for _ in range(n):
        sgn = np.where(rng.random(G) < 0.5, -1.0, 1.0)[gi]; hits += abs((d * sgn).mean()) >= obs - 1e-12
    return (hits + 1) / (n + 1)


def student_arms(teacher_path: Path, feat_clients: Dict[str, Client], meta: dict) -> List[dict]:
    """C2: the ≤10 M student is FEDERATED — each client distils locally (marginal CE on its own labels + KL to the federated teacher's soft
    predictions on its own cached features) and the student is aggregated with the same FedAvg schedule. The centralised student is kept
    only as a clearly labelled oracle upper bound."""
    a, teachers = load_arm_model(teacher_path, meta)
    base = dict(loss="marginal", stage=14, encoder=CFG.encoders[0], model="student", teacher=teachers, teacher_arm=a)
    return [dict(base, name="student_federated_fedavg_kd", algo="fedavg"), dict(base, name="student_centralised_oracle_kd", algo="central")]


def stage14_student_and_stats():
    require_inputs(14)
    out = stage_dir(CFG.ds_final, create=True); runs = all_runs(); summary = {}
    feat_clients, meta = clients_for(CFG.encoders[0]); raw_clients, _ = clients_for(CFG.encoders[0], raw=True)
    print("[stage 14] global Dice is reported three ways: client×structure mean (global_dice), client-mean (global_dice_client_mean), structure-pooled (global_dice_structure_pooled). "
          "FedCVD's Fed-ECHO numbers use a different client set (EchoNet-Dynamic) and are NOT directly comparable.")
    # --- arm table (mean ± sd over seeds) ------------------------------------------------------------------------------------------
    cols = ["global_dice", "global_dice_client_mean", "global_dice_structure_pooled", "worst_client_dice", "heldout_CARDIACNET_RV", "heldout_CARDIACNET_RA", "heldout_CAMUS_LV", "heldout_CAMUS_LA"] + [f"client_dice_{c}" for c in CLIENT_LABELS] + ["n_tx_params", "mb_per_round"]
    table = runs.groupby("arm")[cols].agg(["mean", "std"]); table.columns = [f"{a}_{b}" for a, b in table.columns]; table["n_seeds"] = runs.groupby("arm").seed.nunique(); table["optimizer"] = runs.groupby("arm").optimizer.first(); table.to_csv(out / "arm_table.csv")
    print(table[[c for c in table.columns if c.endswith("_mean")][:8] + ["optimizer"]].round(3).sort_values("global_dice_mean", ascending=False).to_string())
    method = best_feature_arms(1)[0]["name"]; summary["method_arm"] = method; m_runs = runs[runs.arm == method]
    gn = runs[runs.arm == f"{method}__gn_epsinf"]                                  # C4: BN→GN cost shown separately from the DP cost
    summary["bn_vs_gn"] = dict(arm=method, global_dice_bn=float(m_runs.global_dice.mean()), global_dice_gn_nonprivate=float(gn.global_dice.mean()) if len(gn) else np.nan)
    print(f"\n[stage 14] BN→GN cost for {method}: BatchNorm {summary['bn_vs_gn']['global_dice_bn']:.3f} vs GroupNorm non-private {summary['bn_vs_gn']['global_dice_gn_nonprivate']:.3f} global Dice (the ε-curve reference is the GroupNorm row)")
    # --- rounds to 95 % of the centralised-marginal Dice; communication ratio ----------------------------------------------------------
    cm = runs[runs.arm == "central_marginal"].set_index("seed").global_dice; r95 = {}
    for arm_name in sorted(set(runs[runs.stage.isin([11, 12])].arm)):
        vals = []
        for s in CFG.seeds:
            h = stage_dir(CFG.ds_base if arm_name in [a["name"] for a in BASELINE_ARMS] else CFG.ds_method) / f"history_{arm_name}_s{s}.csv"
            if h.exists() and s in cm.index:
                hh = pd.read_csv(h); hit = hh[hh.global_dice >= 0.95 * cm.loc[s]]; vals.append(int(hit["round"].iloc[0]) if len(hit) else np.nan)
        r95[arm_name] = float(np.nanmean(vals)) if vals and not np.all(np.isnan(vals)) else np.nan
    summary["rounds_to_95pct_central_marginal"] = r95
    unet_tx, meth_tx = runs[runs.arm == "unet_fullmodel_fedavg_bg"].n_tx_params.mean(), m_runs.n_tx_params.mean()
    summary["communication"] = dict(unet_tx_params=float(unet_tx), method_tx_params=float(meth_tx), ratio=float(unet_tx / max(meth_tx, 1)), unet_mb_per_round=float(runs[runs.arm == "unet_fullmodel_fedavg_bg"].mb_per_round.mean()), method_mb_per_round=float(m_runs.mb_per_round.mean()))
    summary["gpu_min_per_client"] = json.loads(m_runs.gpu_min_per_client.iloc[0])
    # --- paired cluster bootstrap: method vs local-only per client (own test set) and vs FedInit; Holm over the family -----------------
    ds_of = lambda name: CFG.ds_base if name in [a["name"] for a in BASELINE_ARMS] else CFG.ds_method
    pf_m, pf_local, pf_init = perframe(ds_of(method), method), perframe(CFG.ds_base, "local_only"), perframe(CFG.ds_base, "fedinit_bg"); tests = []
    for c in CLIENT_LABELS:
        for ref_name, ref in (("local_only", pf_local), ("fedinit_bg", pf_init)):
            bs = paired_video_bootstrap(pf_m[pf_m.client == c], ref[ref.client == c], CFG.n_boot); pv = permutation_p(pf_m[pf_m.client == c], ref[ref.client == c], CFG.n_boot)
            tests.append(dict(client=c, comparison=f"{method} − {ref_name}", **bs, perm_p=pv))
    tests = pd.DataFrame(tests); tests["excludes_zero"] = (tests.ci_lo > 0) | (tests.ci_hi < 0); tests["perm_p_holm"] = holm(tests.perm_p.tolist())   # D8: six comparisons, Holm-corrected
    tests.to_csv(out / "paired_tests.csv", index=False); print("\n", tests.round(4).to_string(index=False))
    # --- privacy curve and attack audit -----------------------------------------------------------------------------------------------------
    priv = runs[runs.stage == 13].copy(); priv["eps"] = priv.dp_eps_target.fillna(np.inf)
    curve = priv.groupby(["arm", "eps"]).agg(global_dice=("global_dice", "mean"), global_sd=("global_dice", "std"), worst=("worst_client_dice", "mean"), eps_spent=("eps_spent", "first")).reset_index(); curve.to_csv(out / "utility_eps_curve.csv", index=False)
    mia = pd.read_csv(stage_dir(CFG.ds_priv) / "membership_inference.csv"); summary["mia"] = mia[["arm", "eps_target", "mean_loss_threshold_auc", "mean_shadow_auc"]].to_dict("records")
    summary["dp_accounting"] = dict(unit=CFG.dp_unit, frames_per_video=CFG.dp_frames_per_video, clip_norm=CFG.dp_clip, accountant="RDP (Opacus)", delta="1/N_videos per client",
                                    per_arm={r.arm: dict(eps_target=float(r.dp_eps_target), eps_spent=json.loads(r.eps_spent), params=json.loads(r.dp_params)) for _, r in priv[priv.dp_eps_target.notna() & (priv.seed == 0)].iterrows()})
    conf = pd.read_csv(stage_dir(CFG.ds_priv) / "conformal_margins.csv"); summary["conformal"] = {f"{a}|{c}": float(v) for (a, c), v in conf.groupby(["arm", "client"]).test_coverage.mean().items()}
    # --- RV/LV area ratio from the federated masks as a PAH discriminator on the CardiacNet held-out fold, against the RHC label ------------
    cn = pf_m[(pf_m.client == "CARDIACNET")].drop_duplicates(["id"]); vid = cn.sort_values("area_lv", ascending=False).drop_duplicates("video")   # ED proxy: frame with the largest LV
    ratio = (vid.area_rv / (vid.area_lv + 1e-4)).to_numpy(); y = vid.pah.to_numpy(); k = ~np.isnan(y)
    summary["rv_lv_ratio_pah_auroc"] = float(roc_auc_score(y[k], ratio[k])) if len(np.unique(y[k])) == 2 else np.nan
    # --- federated student distilled from the federated model (C2); centralised student = oracle upper bound ----------------------------
    slog, sman = RunLog(out), RunManifest(out / "runs_manifest.json"); rounds_bak = CFG.rounds; CFG.rounds = CFG.student_rounds
    for a in student_arms(stage_dir(ds_of(method)) / f"model_{method}_s0.pt", feat_clients, meta):
        run_and_log({**a, "seed": 0}, out, slog, sman, raw_clients, meta, save_model=True)
    CFG.rounds = rounds_bak; srows = slog.read(); summary["student"] = {r.arm: dict(params=int(r.params_model), global_dice=float(r.global_dice), heldout_CARDIACNET_RV=float(r.heldout_CARDIACNET_RV), n_tx_params=int(r.n_tx_params), mb_per_round=float(r.mb_per_round)) for _, r in srows.iterrows()}
    for k_, v in summary["student"].items(): print(f"[stage 14] {k_}: {v['params']/1e6:.2f} M params | global Dice {v['global_dice']:.3f} | CardiacNet RV {v['heldout_CARDIACNET_RV']:.3f} | tx {v['n_tx_params']/1e6:.2f} M params/round")
    # --- withdrawal conditions -----------------------------------------------------------------------------------------------------------------
    loc = runs[runs.arm == "local_only"]; init = runs[runs.arm == "fedinit_bg"]
    beats_local = {c: bool(m_runs[f"client_dice_{c}"].mean() > loc[f"client_dice_{c}"].mean()) for c in CLIENT_LABELS}
    beats_fedinit = bool(m_runs.global_dice.mean() > init.global_dice.mean())
    best_cfg = best_feature_arms(1)[0]["name"]; np_ = curve[(curve.arm == f"{best_cfg}__gn_epsinf")].global_dice; e8 = curve[(curve.arm == f"{best_cfg}__gn_eps{CFG.dp_eps[-1]}")].global_dice
    findings = dict(method_claim=dict(withdraw=not (all(beats_local.values()) and beats_fedinit), beats_local_on_own_test=beats_local, beats_fedinit=beats_fedinit, method=method),
                    communication_claim=dict(withdraw=bool(summary["communication"]["ratio"] < 10), ratio=summary["communication"]["ratio"]),
                    privacy_framing=dict(withdraw=bool(len(np_) and len(e8) and float(np_.iloc[0]) - float(e8.iloc[0]) > 0.03), dice_nonprivate=float(np_.iloc[0]) if len(np_) else np.nan, dice_eps8=float(e8.iloc[0]) if len(e8) else np.nan),
                    benchmark_extension_claim=dict(stands=True, note="first federation with RV and RA; partially labelled echo FL beyond FedCVD's baselines; split manifests published with the cache"))
    summary["findings"] = findings; summary["utility_eps_curve"] = curve.to_dict("records")
    # --- figures --------------------------------------------------------------------------------------------------------------------------------
    fig, ax = plt.subplots(1, 4, figsize=(21, 4.5))
    for arm_name, g in curve.groupby("arm"):
        g = g.sort_values("eps"); x = [str(e) for e in g.eps]; ax[0].errorbar(x, g.global_dice, yerr=g.global_sd.fillna(0), marker="o", label=arm_name.split("__")[0])
    ax[0].set_title("Utility–ε (global Dice)"); ax[0].set_xlabel("ε (δ = 1/N_client)"); ax[0].legend(fontsize=7)
    for arm_name in ["fedavg_bg", "fedinit_bg", "fedavg_marginal", "fedavg_marginal_condist", "fedavg_marginal_condist_fedbn", "central_marginal"]:
        h = stage_dir(ds_of(arm_name)) / f"history_{arm_name}_s0.csv"
        if h.exists(): hh = pd.read_csv(h); ax[1].plot(hh["round"], hh.global_dice, label=arm_name)
    ax[1].set_title("Global Dice vs round (seed 0)"); ax[1].set_xlabel("round"); ax[1].legend(fontsize=7)
    cl = runs[runs.arm.isin(["local_only", "fedavg_bg", "fedinit_bg", method])].groupby("arm")[[f"client_dice_{c}" for c in CLIENT_LABELS]].mean(); cl.T.plot.bar(ax=ax[2], rot=0); ax[2].set_title("Per-client Dice (own test set)")
    ax[3].plot(mia.eps_target.fillna(np.inf).astype(str), mia.mean_loss_threshold_auc, "o-", label="loss threshold"); ax[3].plot(mia.eps_target.fillna(np.inf).astype(str), mia.mean_shadow_auc, "s-", label="shadow"); ax[3].axhline(0.5, color="k", ls=":"); ax[3].set_title("Membership-inference AUROC vs ε"); ax[3].legend(fontsize=8)
    plt.tight_layout(); plt.savefig(out / "k2_figures.png", dpi=110); plt.close()
    (out / "summary.json").write_text(json.dumps(summary, indent=1, default=lambda o: float(o) if isinstance(o, (np.floating, np.integer)) else str(o)))
    write_version(CFG.ds_final)
    print("\nFindings:"); [print(f"  {k}: {json.dumps(v, default=str)[:300]}") for k, v in findings.items()]
    print(f"[stage 14] summary → {out/'summary.json'}")


if 14 in CFG.stages:
    stage14_student_and_stats()

[stage 14] depends on: ['fed-features-dinov3-v1', 'fed-baselines-v1', 'fed-method-v1', 'fed-privacy-v1']
   fed-features-dinov3-v1 -> k2_work/fed-features-dinov3-v1  (2026-09-29T13:37:27)
   fed-baselines-v1     -> k2_work/fed-baselines-v1  (2026-09-29T13:39:42)
   fed-method-v1        -> k2_work/fed-method-v1  (2026-09-29T13:43:34)
   fed-privacy-v1       -> k2_work/fed-privacy-v1  (2026-09-29T13:47:33)


[stage 14] global Dice is reported three ways: client×structure mean (global_dice), client-mean (global_dice_client_mean), structure-pooled (global_dice_structure_pooled). FedCVD's Fed-ECHO numbers use a different client set (EchoNet-Dynamic) and are NOT directly comparable.
                                        global_dice_mean  global_dice_client_mean_mean  global_dice_structure_pooled_mean  worst_client_dice_mean  heldout_CARDIACNET_RV_mean  heldout_CARDIACNET_RA_mean  heldout_CAMUS_LV_mean  heldout_CAMUS_LA_mean                optimizer
arm                                                                                                                                                                                                                                                                             
central_marginal                                   0.762                         0.737                              0.758                   0.665                       0.759     


[stage 14] BN→GN cost for fedavg_marginal_condist: BatchNorm 0.611 vs GroupNorm non-private 0.587 global Dice (the ε-curve reference is the GroupNorm row)



     client                           comparison   delta   ci_lo  ci_hi  n_videos  perm_p  excludes_zero  perm_p_holm
     CAMUS fedavg_marginal_condist − local_only  0.1889  0.1553 0.2224         2  0.4677           True          1.0
     CAMUS fedavg_marginal_condist − fedinit_bg  0.1399  0.1109 0.1688         2  0.4677           True          1.0
     HMCQU fedavg_marginal_condist − local_only -0.0453 -0.1089 0.0184         2  1.0000          False          1.0
     HMCQU fedavg_marginal_condist − fedinit_bg  0.4934  0.4140 0.5728         2  0.4677           True          1.0
CARDIACNET fedavg_marginal_condist − local_only  0.4526  0.4177 0.4712         3  0.2189           True          1.0
CARDIACNET fedavg_marginal_condist − fedinit_bg  0.4167  0.3860 0.4440         3  0.2189           True          1.0


[student_federated_fedavg_kd        s0] global Dice 0.626 | worst client 0.198 | CardiacNet RV 0.747 RA 0.256 | tx 1.95 M params, 44.6 MB/round (12s)


[student_centralised_oracle_kd      s0] global Dice 0.917 | worst client 0.856 | CardiacNet RV 0.930 RA 0.870 | tx 0.00 M params, 0.0 MB/round (10s)
[stage 14] student_centralised_oracle_kd: 1.94 M params | global Dice 0.917 | CardiacNet RV 0.930 | tx 0.00 M params/round
[stage 14] student_federated_fedavg_kd: 1.94 M params | global Dice 0.626 | CardiacNet RV 0.747 | tx 1.95 M params/round



Findings:
  method_claim: {"withdraw": true, "beats_local_on_own_test": {"CAMUS": true, "HMCQU": false, "CARDIACNET": true}, "beats_fedinit": true, "method": "fedavg_marginal_condist"}
  communication_claim: {"withdraw": false, "ratio": 13.426836811206595}
  privacy_framing: {"withdraw": true, "dice_nonprivate": 0.5871668240976295, "dice_eps8": 0.06714403353064921}
  benchmark_extension_claim: {"stands": true, "note": "first federation with RV and RA; partially labelled echo FL beyond FedCVD's baselines; split manifests published with the cache"}
[stage 14] summary → k2_work/k2-final-v1/summary.json
